# Madagascar — national pluvial flood model: input pre-processing

**Model engine:** [SFINCS](https://sfincs.readthedocs.io) (Super-Fast INundation of CoastS), rain-on-grid mode
**Domain:** Madagascar, national coverage (subdivided into pluvial catchments)
**Notebook role:** turn public global datasets into a documented, reproducible set of SFINCS-ready inputs.

---

## Purpose and scope

This notebook is the **single entry point** for pre-processing every input of the national-scale
pluvial (rain-on-grid) flood model. Running it top to bottom on a clean machine reproduces the
complete input set in `Data/processed/`, plus a HydroMT-SFINCS data catalogue that points at those files.

Design rules followed throughout:

1. **No manual downloads.** Every dataset is pulled programmatically (HTTP range requests, `/vsicurl`,
   WCS, or a repository API). The only pre-existing local inputs are the HydroBASINS polygons and the
   GPM daily rainfall cube (see *Local inputs* below).
2. **Idempotent and resumable.** Each step skips work whose output already exists on disk
   (`OVERWRITE = False`). Interrupted downloads can simply be re-run.
3. **Native CRS is preserved.** All static rasters are written in **EPSG:4326**, their native grid.
   Reprojection to a metric model CRS is deliberately *not* done here: it is a per-model-domain
   decision made by HydroMT-SFINCS when each SFINCS grid is built (Madagascar straddles
   UTM 38S and 39S, so there is no single sensible national metric grid).
4. **Provenance is recorded.** Section 5.2 writes a manifest with source URL, access date, size
   and checksum for every output.

---

## Input datasets

| # | Input | Type | Dataset / source | Native res. | Access route | Role in the pluvial model | Output written here |
|---|-------|------|------------------|-------------|--------------|---------------------------|---------------------|
| 1 | Modelling domain | Static | HydroBASINS v1c, level 07 (Africa) | vector | local file (HydroSHEDS) | Splits the country into independent pluvial catchments = SFINCS model domains | `processed/domain/pluvial_catchments_mdg.gpkg` |
| 2 | Topography (DTM) | Static | **FABDEM V1-2**, 30 m, EGM2008 orthometric | 1 arcsec | HTTP range reads into the University of Bristol tile archives | Surface routing, depression storage, subgrid elevation detail | `raw/fabdem/*.tif` + `processed/topography/fabdem_mdg.vrt` |
| 3 | Land use / land cover | Static | **Copernicus Global Land Cover** (VITO PROBA-V LC100 v3.0.1, 2019) | 100 m | `/vsicurl` windowed read of the global COG on Zenodo | Roughness and infiltration parameterisation | `processed/lulc/vito_lc100_2019_mdg.tif` |
| 4 | Curve Number | Static | **GCN250** (Jaafar et al. 2019), ARC I / II / III | 250 m | figshare REST API + cached download | SCS-CN infiltration / effective rainfall | `processed/curve_number/gcn250_arc{i,ii,iii}_mdg.tif` |
| 5 | Soil texture / Hydrologic Soil Group | Static | **SoilGrids 2.0** sand & clay, 0–100 cm | 250 m | ISRIC WCS 2.0.1 `GetCoverage` | Green-Ampt parameters; independent check on GCN250 | `processed/soil/hsg_mdg.tif` (+ sand/clay/texture) |
| 6 | Rainfall field | Dynamic | **GPM IMERG Final Run daily** (0.1°) + **IBTrACS v04r00** cyclone best tracks | 0.1°, daily | local cube + direct netCDF download from NOAA NCEI | Rain-on-grid forcing; TC attribution; design hyetographs | `processed/rainfall/*.nc`, `*.csv` |

### Deliberately not produced here

Two inputs listed in the model specification are left to the model build:

* **Surface roughness (Manning's *n*)** — a direct look-up on dataset 3. The look-up table
  (`vito_mapping.csv`, Deltares) is loaded and shown in section 3.2 so it travels with the data,
  but the roughness grid itself is generated inside the SFINCS model build, where the target grid is known.
* **Observation points** — set arbitrarily per model domain after the grids exist.

---

## Local inputs (must be present before running)

| File | Description | Where it comes from |
|------|-------------|---------------------|
| `Data/hybas_af_lev07_v1c_mdg.geojson` | HydroBASINS v1c level-07 basins clipped to Madagascar | [hydrosheds.org](https://www.hydrosheds.org/products/hydrobasins) — `hybas_af_lev07_v1c`, clipped to the national outline |
| `Data/gpm_daily_precipitation_pp.nc` | GPM IMERG daily precipitation (mm/day), 0.1°, 1998-01-01 → 2024-09-30, clipped to 42.4–50.8 °E / 26.0–11.1 °S | GES DISC `GPM_3IMERGDF` v06/v07, pre-aggregated |

---

## Output tree

```
Execution/
├── Data/
│   ├── raw/                        # untouched source data, cached (large, git-ignored)
│   │   ├── fabdem/                 # 83 x 1-degree FABDEM tiles (~1.4 GB)
│   │   └── gcn250/                 # global GCN250 GeoTIFFs (~0.6 GB per ARC)
│   └── processed/                  # SFINCS-ready inputs
│       ├── domain/                 # pluvial catchments
│       ├── topography/             # FABDEM VRT (+ optional coarse overview)
│       ├── lulc/                   # VITO land cover
│       ├── curve_number/           # GCN250 ARC I/II/III
│       ├── soil/                   # SoilGrids sand/clay, USDA texture, HSG
│       ├── rainfall/               # TC catalogue, event rainfall, design hyetographs
│       ├── sfincs_data_catalog.yml # HydroMT-SFINCS entry point
│       └── provenance.json         # manifest: source, date, size, checksum
└── Figures/                        # QC figures produced by this notebook
```

---

## Software environments

The notebook needs **one** environment, and every package in it installs from a binary wheel —
nothing is compiled, and no external binary has to be on `PATH`:

| Requirement | Why |
|---|---|
| `numpy` `pandas` `geopandas` `shapely` `pyproj` | vector work, catchments, swaths |
| `rasterio` | **all** raster I/O, including the VRT mosaic and the resampling in section 3.1 — its bundled GDAL is the only GDAL needed |
| `xarray` **+ `netcdf4`** | the GPM cube and every netCDF written here |
| `requests` | FABDEM range reads, figshare, SoilGrids WCS, IBTrACS |
| `matplotlib`, `contextily` | QC figures (`contextily` optional — figures degrade to no basemap) |
| `scipy`, `tqdm` | optional |

Three dependencies that are conventional for this kind of work are **deliberately avoided**, because
each one is a recurring source of broken environments:

* **`osgeo` / `gdal` Python bindings** — a compiled extension that must match the system GDAL
  exactly; `pip install gdal` frequently fails to build. Section 3.1 writes its VRT mosaic as XML
  and resamples with `rasterio.vrt.WarpedVRT` instead, then verifies the mosaic against the source
  tiles pixel for pixel.
* **The `gdal*` command-line tools** — not needed either, so a half-linked GDAL install (e.g. a
  Homebrew `gdal` whose `libpoppler` dependency has moved on) cannot break this notebook.
* **CLIMADA** — section 4.2 reads the IBTrACS netCDF directly from NOAA NCEI (7 MB).

With conda:

```bash
conda create -n mdg_pluvial -c conda-forge python=3.12 rasterio geopandas xarray netcdf4     matplotlib contextily requests pyproj shapely pandas numpy scipy tqdm ipykernel
conda activate mdg_pluvial
python -m ipykernel install --user --name mdg_pluvial --display-name "Python (mdg_pluvial)"
```

If `conda` itself is broken — a corrupted base environment can make `conda create` fail before it
even parses the command — a **plain venv needs no conda at all**, and is how this notebook was
validated:

```bash
python3.12 -m venv .venv-mdg_pluvial
.venv-mdg_pluvial/bin/python -m pip install --upgrade pip
.venv-mdg_pluvial/bin/python -m pip install numpy pandas geopandas rasterio xarray netCDF4     matplotlib contextily requests pyproj shapely scipy tqdm ipykernel
.venv-mdg_pluvial/bin/python -m ipykernel install --user --name mdg_pluvial     --display-name "Python (mdg_pluvial)"
```

Do **not** add `gdal` to either command's package list. It is unnecessary here, and it is the one
package likely to fail.

Optional, for the SFINCS model build that consumes these outputs:
`pip install hydromt hydromt_sfincs`.

The code is kept compatible with **Python 3.8 → 3.13**, numpy 1.23 → 2.5, pandas 1.5 → 3.0 and
rasterio 1.2 → 1.5. It was validated end to end on Python 3.12 / numpy 2.5 / pandas 3.0 /
rasterio 1.5, and on Python 3.8 / numpy 1.23 / pandas 1.5 / rasterio 1.2 — so an older project
environment does not need upgrading. The one thing an environment must not be missing is
**`netcdf4`**: without it `xr.open_dataset` cannot read the GPM cube and section 4 fails
immediately.

## Runtime and disk budget

| Section | Network | Disk | Wall time (first run) |
|---------|---------|------|-----------------------|
| 2 Modelling domain | – | 8 MB | < 1 min |
| 3.1 FABDEM tiles | ~1.4 GB | 1.4 GB | 20–40 min |
| 3.1.2 DEM overview (optional) | – | 0.2 GB | 3–8 min |
| 3.2 VITO LULC | ~0.1 GB | 25 MB | 1–2 min |
| 3.3 GCN250 | 0.6–1.9 GB | 0.6–1.9 GB | 10–30 min |
| 3.4 SoilGrids | ~0.1 GB | 60 MB | 1–2 min |
| 4 Rainfall | 7 MB (IBTrACS, once) | 6 MB | 3–5 min |
| 5 Catalogue and provenance | – | < 1 MB | < 1 min |

Re-runs with all caches present: **< 3 minutes**.

---

## Licences and citations

Check these before any redistribution or commercial use of derived products.

| Dataset | Licence | Citation |
|---------|---------|----------|
| FABDEM V1-2 | **CC BY-NC-SA 4.0 — non-commercial, share-alike.** Commercial use requires a licence from Fathom (`fabdem@fathom.global`). | Hawker, L. et al. (2022) *A 30 m global map of elevation with forests and buildings removed*, Environ. Res. Lett. 17, 024016 |
| Copernicus Global Land Cover | Free and open (Copernicus) | Buchhorn, M. et al. (2020) *Copernicus Global Land Service: Land Cover 100 m, epoch 2019*, Zenodo, doi:10.5281/zenodo.3939050 |
| GCN250 | CC BY 4.0 | Jaafar, H. et al. (2019) *GCN250, new global gridded curve numbers for hydrologic modeling and design*, Scientific Data 6, 145 |
| SoilGrids 2.0 | CC BY 4.0 | Poggio, L. et al. (2021) *SoilGrids 2.0*, SOIL 7, 217–240 |
| GPM IMERG | Free and open (NASA) | Huffman, G.J. et al. (2019) *GPM IMERG Final Precipitation L3 1 day 0.1 deg*, GES DISC |
| HydroBASINS v1c | Free for non-commercial and commercial use with attribution | Lehner, B. & Grill, G. (2013) *Global river hydrography and network routing*, Hydrol. Process. 27, 2171–2186 |
| IBTrACS v04r00 | Free and open (NOAA NCEI) | Knapp, K.R. et al. (2010) *The International Best Track Archive for Climate Stewardship*, Bull. Am. Meteorol. Soc. 91, 363–376 |

> **FABDEM is non-commercial.** For a commercial engagement, substitute Copernicus DEM GLO-30
> (open) or a licensed FABDEM copy. Only the source URL in section 4.1 changes; the rest of the
> pipeline is unaffected.

---

# 1. Set-up

## 1.1 Imports and environment report

Nothing below is imported lazily except `climada` (section 6.1), which lives in a separate
environment. The cell prints the resolved versions so a run can be reproduced later.

In [ ]:
import hashlib
import io
import json
import math
import os
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
from xml.sax.saxutils import escape

import numpy as np
import pandas as pd
import requests

import geopandas as gpd
import rasterio
import xarray as xr
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window, from_bounds
from shapely.geometry import box

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap

try:
    import contextily as cx
    HAS_BASEMAP = True
except Exception as exc:                                   # pragma: no cover
    HAS_BASEMAP = False
    print(f"contextily unavailable ({exc}) - figures will be drawn without a basemap")

plt.rcParams.update({"font.size": 12, "figure.dpi": 100})

print("Environment report")
print("-" * 60)
print(f"{'python':<14} {sys.version.split()[0]}  ({sys.executable})")
for _mod in (np, pd, gpd, rasterio, xr, requests):
    print(f"{_mod.__name__:<14} {getattr(_mod, '__version__', 'n/a')}")
print(f"{'GDAL':<14} {rasterio.__gdal_version__}  (bundled with rasterio)")
print(f"{'run started':<14} {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

## 1.2 Configuration

Everything a re-run might need to change lives in this cell. `EXEC_DIR` is discovered from the
notebook's working directory (the folder holding both `Data/` and `Scripts/`) so the notebook carries
no machine-specific absolute paths; override it with the `MDG_EXEC_DIR` environment variable if you
keep the data elsewhere.

In [ ]:
# --------------------------------------------------------------- project paths
def _find_exec_dir(start: Path) -> Path:
    """Walk up from *start* until a folder holding both Data/ and Scripts/ is found."""
    for cand in (start, *start.parents):
        if (cand / "Data").is_dir() and (cand / "Scripts").is_dir():
            return cand
    raise FileNotFoundError(
        "Could not locate the Execution folder (expected to contain Data/ and Scripts/). "
        "Set the MDG_EXEC_DIR environment variable to point at it."
    )


EXEC_DIR = (
    Path(os.environ["MDG_EXEC_DIR"]).expanduser().resolve()
    if os.environ.get("MDG_EXEC_DIR")
    else _find_exec_dir(Path.cwd().resolve())
)

DATA_DIR = EXEC_DIR / "Data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
FIG_DIR = EXEC_DIR / "Figures"

DIRS = {
    "domain": PROC_DIR / "domain",
    "topography": PROC_DIR / "topography",
    "lulc": PROC_DIR / "lulc",
    "curve_number": PROC_DIR / "curve_number",
    "soil": PROC_DIR / "soil",
    "rainfall": PROC_DIR / "rainfall",
    "fabdem_raw": RAW_DIR / "fabdem",
    "gcn250_raw": RAW_DIR / "gcn250",
}
for _d in (RAW_DIR, PROC_DIR, FIG_DIR, *DIRS.values()):
    _d.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------- local inputs
HYBAS_FILE = DATA_DIR / "hybas_af_lev07_v1c_mdg.geojson"
GPM_FILE = DATA_DIR / "gpm_daily_precipitation_pp.nc"

# ----------------------------------------------------------------- behaviour
OVERWRITE = False        # True -> ignore cached outputs and rebuild everything
MAKE_FIGURES = True      # QC figures (adds a few minutes for the DEM overview)

# ------------------------------------------------------------ domain settings
MIN_CATCHMENT_AREA_KM2 = 50.0   # drops offshore islets; keeps Nosy Be (~320) and Sainte Marie (~150)
AOI_BUFFER_DEG = 0.10           # halo on the national bbox, so catchment edges are never clipped
AREA_CRS = "EPSG:32738"         # UTM 38S - used ONLY for area computations
WORK_CRS = "EPSG:4326"          # CRS of every raster written by this notebook

# ---------------------------------------------------------- static data flags
GCN250_ARCS = ("ARCII",)        # ("ARCI", "ARCII", "ARCIII") for the full antecedent-moisture set
MAKE_DEM_OVERVIEW = True        # coarse national DEM mosaic, for QC plots only
DEM_OVERVIEW_RES_DEG = 0.0025   # ~250 m

SOILGRIDS_DEPTHS = ("0-5cm", "5-15cm", "15-30cm", "30-60cm", "60-100cm")
SOILGRIDS_WEIGHTS = (5, 10, 15, 30, 40)      # layer thickness in cm -> depth-weighted mean

# ------------------------------------------------------------ rainfall config
TC_SEARCH_RADIUS_KM = 500       # TC-attribution radius around the track centre
SWATH_RADIUS_KM = 300           # swath radius for event maps
EVENT_PAD_DAYS = 1              # days kept either side of the best track
EXPORT_EVENTS = ("ENAWO", "BATSIRAI", "GAFILO")   # events exported as SFINCS rain-on-grid forcing

# --------------------------------------------------------------- http session
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "mdg-pluvial-preprocessing/1.0"})
HTTP_TIMEOUT = 180

print(f"EXEC_DIR : {EXEC_DIR}")
print(f"outputs  : {PROC_DIR}")
missing = [str(p.relative_to(EXEC_DIR)) for p in (HYBAS_FILE, GPM_FILE) if not p.exists()]
print("local inputs:", "all present" if not missing else f"MISSING -> {missing}")

## 1.3 Helper functions

Generic utilities shared by the dataset sections:

* `HTTPRangeFile` — a seekable file-like object backed by HTTP `Range` requests. It lets `zipfile`
  read the central directory of a remote archive and extract single members, which is what makes the
  FABDEM tile-level download in section 4.1 possible without pulling the 861 MB archives whole.
* `http_download` — streamed download with a `.part` file, so an interrupted run never leaves a
  truncated output behind.
* `clip_remote_raster` / `write_raster` — windowed read of a (possibly remote) raster over the AOI
  and write-out as a tiled, compressed GeoTIFF.
* `describe_raster` — the QC print used after every dataset is written.
* `register_output` — appends to the provenance manifest assembled in section 7.

In [ ]:
# ------------------------------------------------------- provenance manifest
OUTPUTS: List[dict] = []


def register_output(path: Path, *, dataset: str, source: str, note: str = "") -> Path:
    """Record a produced file in the provenance manifest (see section 7)."""
    path = Path(path)
    entry = {
        "dataset": dataset,
        "path": str(path.relative_to(EXEC_DIR)) if EXEC_DIR in path.parents else str(path),
        "source": source,
        "note": note,
        "size_mb": round(path.stat().st_size / 1e6, 2) if path.exists() else None,
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    OUTPUTS[:] = [e for e in OUTPUTS if e["path"] != entry["path"]] + [entry]
    return path


def sha256(path: Path, chunk: int = 8 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def human(n_bytes: float) -> str:
    """Decimal file size (1 MB = 1e6 B), to match how the data portals quote sizes."""
    for unit in ("B", "kB", "MB", "GB", "TB"):
        if abs(n_bytes) < 1000 or unit == "TB":
            return f"{n_bytes:,.1f} {unit}"
        n_bytes /= 1000


# ------------------------------------------------- remote random-access reads
class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP byte ranges.

    Wrap in ``io.BufferedReader`` and hand to ``zipfile.ZipFile`` to list and extract
    individual members of a remote archive without downloading the whole thing.
    """

    def __init__(self, url: str, session: Optional[requests.Session] = None, timeout: int = HTTP_TIMEOUT):
        self.session = session or requests.Session()
        self.timeout = timeout
        head = self.session.head(url, allow_redirects=True, timeout=timeout)
        head.raise_for_status()
        self.url = head.url                       # follow redirects once, then reuse
        self.size = int(head.headers["Content-Length"])
        if head.headers.get("Accept-Ranges") != "bytes":
            raise RuntimeError(f"server does not advertise byte ranges: {url}")
        self._pos = 0
        self.bytes_read = 0

    # --- io.RawIOBase plumbing -------------------------------------------------
    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self._pos

    def seek(self, offset: int, whence: int = io.SEEK_SET) -> int:
        self._pos = {io.SEEK_SET: offset,
                     io.SEEK_CUR: self._pos + offset,
                     io.SEEK_END: self.size + offset}[whence]
        return self._pos

    def readinto(self, buf) -> int:
        data = self.read(len(buf))
        buf[: len(data)] = data
        return len(data)

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self._pos
        if n == 0 or self._pos >= self.size:
            return b""
        end = min(self._pos + n, self.size) - 1
        resp = self.session.get(self.url, headers={"Range": f"bytes={self._pos}-{end}"},
                                timeout=self.timeout)
        resp.raise_for_status()
        data = resp.content
        self._pos += len(data)
        self.bytes_read += len(data)
        return data


def remote_zip(url: str, buffer_mb: int = 4) -> Tuple[zipfile.ZipFile, HTTPRangeFile]:
    """Open a remote ZIP for member-level access. Returns (ZipFile, underlying range file)."""
    raw = HTTPRangeFile(url, session=SESSION)
    return zipfile.ZipFile(io.BufferedReader(raw, buffer_size=buffer_mb << 20)), raw


def http_download(url: str, dest: Path, *, overwrite: bool = False, chunk_mb: int = 8) -> Path:
    """Stream *url* to *dest* via a .part file. Skips the download if *dest* already exists."""
    dest = Path(dest)
    if dest.exists() and not overwrite:
        print(f"  cached   {dest.name} ({human(dest.stat().st_size)})")
        return dest
    part = dest.with_suffix(dest.suffix + ".part")
    t0 = time.time()
    with SESSION.get(url, stream=True, timeout=HTTP_TIMEOUT) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("Content-Length", 0))
        done = 0
        with open(part, "wb") as fh:
            for block in resp.iter_content(chunk_size=chunk_mb << 20):
                fh.write(block)
                done += len(block)
                if total:
                    print(f"\r  download {dest.name}: {100 * done / total:5.1f}% "
                          f"({human(done)} / {human(total)})", end="")
    print(f"\r  download {dest.name}: {human(done)} in {time.time() - t0:.0f}s" + " " * 20)
    part.replace(dest)
    return dest


# ------------------------------------------------------------- raster helpers
COG_PROFILE = dict(driver="GTiff", tiled=True, blockxsize=512, blockysize=512,
                   compress="deflate", predictor=2, BIGTIFF="IF_SAFER")


def write_raster(path: Path, array: np.ndarray, transform, *, crs=WORK_CRS,
                 nodata=None, dtype=None, overviews: bool = True,
                 descriptions: Optional[Sequence[str]] = None) -> Path:
    """Write a 2-D or 3-D array as a tiled, deflate-compressed GeoTIFF with overviews."""
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[None, :, :]
    dtype = dtype or array.dtype
    if np.dtype(dtype) == np.float64:                 # float32 is plenty for every layer here
        dtype = "float32"
    profile = dict(COG_PROFILE, height=array.shape[1], width=array.shape[2],
                   count=array.shape[0], dtype=dtype, crs=crs, transform=transform)
    if nodata is not None:
        profile["nodata"] = nodata
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array.astype(dtype))
        if descriptions:
            for i, name in enumerate(descriptions, start=1):
                dst.set_band_description(i, name)
        if overviews:
            factors = [f for f in (2, 4, 8, 16, 32) if min(array.shape[1:]) // f >= 64]
            if factors:
                dst.build_overviews(factors, Resampling.nearest)
                dst.update_tags(ns="rio_overview", resampling="nearest")
    return path


def clip_remote_raster(src_uri: str, bounds: Tuple[float, float, float, float],
                       band: int = 1) -> Tuple[np.ndarray, object, dict]:
    """Windowed read of *src_uri* (local path, URL or /vsicurl/...) over *bounds* (W, S, E, N).

    Returns (array, transform, source profile). The window is snapped outwards to whole
    source pixels, so no resampling happens - the output stays on the source grid.
    The floor/ceil arithmetic is done by hand rather than with ``Window.round_*``, whose
    rounding mode has changed across rasterio versions.
    """
    with rasterio.open(src_uri) as src:
        w = from_bounds(*bounds, transform=src.transform)
        col_off, row_off = math.floor(w.col_off), math.floor(w.row_off)
        window = Window(col_off, row_off,
                        math.ceil(w.width + (w.col_off - col_off)),
                        math.ceil(w.height + (w.row_off - row_off)))
        window = window.intersection(Window(0, 0, src.width, src.height))
        array = src.read(band, window=window)
        transform = src.window_transform(window)
        meta = dict(crs=src.crs, nodata=src.nodata, dtype=src.dtypes[band - 1],
                    res=src.res, source_shape=(src.height, src.width))
    return array, transform, meta


def describe_raster(path: Path, *, label: str = "", sample_max: int = 4_000_000) -> None:
    """QC print: grid definition plus a valid-data summary read from the coarsest overview."""
    with rasterio.open(path) as src:
        decim = max(1, int(math.sqrt(src.width * src.height / sample_max)))
        arr = src.read(1, out_shape=(src.height // decim or 1, src.width // decim or 1),
                       resampling=Resampling.nearest).astype("float64")
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
        valid = np.isfinite(arr)
        print(f"{label or path.name}")
        print(f"  path      {path.relative_to(EXEC_DIR) if EXEC_DIR in path.parents else path}")
        print(f"  size      {human(path.stat().st_size)}  |  {src.count} band(s) {src.dtypes[0]}")
        print(f"  grid      {src.width} x {src.height} px  @ {src.res[0]:.8g} deg  ({src.crs})")
        print(f"  bounds    {tuple(round(v, 4) for v in src.bounds)}  |  nodata {src.nodata}")
        if valid.any():
            print(f"  values    valid {100 * valid.mean():.1f}%  "
                  f"min {np.nanmin(arr):.3f}  mean {np.nanmean(arr):.3f}  max {np.nanmax(arr):.3f}"
                  f"   (sampled 1:{decim})")
        else:
            print("  values    NO VALID DATA - check the AOI and the source grid")


def add_basemap(ax, source=None, crs=4326):
    if not HAS_BASEMAP:
        return
    try:
        cx.add_basemap(ax, crs=crs, attribution=False,
                       source=source or cx.providers.Esri.WorldShadedRelief)
    except Exception as exc:                                # offline / tile server down
        print(f"  (basemap skipped: {exc})")


def savefig(name: str) -> Path:
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, format="png", dpi=200, bbox_inches="tight")
    return path

---

# 2. Modelling domain — pluvial catchment delineation

A single SFINCS grid covering Madagascar at 30 m would be ~1.9 x 10<sup>9</sup> cells, so the country is
split into hydrologically independent **pluvial catchments**, each becoming its own SFINCS domain.

The delineation uses **HydroBASINS v1c level 07** (Pfafstetter-coded sub-basins, ~1 000 km² median
area for Africa), which is the level where basins are small enough for a tractable rain-on-grid
model yet large enough to keep the number of models manageable.

Processing steps:

1. **Explode** multipolygons into single-part polygons. HydroBASINS features can bundle a mainland
   basin with its offshore islets; each part must be judged separately.
2. **Compute areas** in UTM 38S (EPSG:32738), a metric CRS covering most of Madagascar. Areas are
   only used for filtering, so the modest distortion on the east coast is irrelevant.
3. **Filter** on `MIN_CATCHMENT_AREA_KM2` (50 km²). This drops the offshore islets while keeping
   Nosy Be (~320 km²) and Île Sainte-Marie (~150 km²). Inspect the area distribution below before
   changing the threshold.
4. **Assign a stable `catchment_id`** — the key that ties every downstream SFINCS model, forcing file
   and result back to a domain. It is derived from `hybas_id`, so it does not shift if the
   threshold changes.

In [ ]:
gdf_hybas = gpd.read_file(HYBAS_FILE)
print(f"HydroBASINS input : {len(gdf_hybas)} features, CRS {gdf_hybas.crs}")

# 1. explode multipolygons into single-part polygons
gdf_exp = gdf_hybas.explode(index_parts=True).reset_index(drop=True)

# 2. metric areas (UTM 38S) - filtering only
gdf_exp["area_km2"] = gdf_exp.to_crs(AREA_CRS).area / 1e6

# 3. area filter
gdf_catch = gdf_exp[gdf_exp["area_km2"] >= MIN_CATCHMENT_AREA_KM2].copy()

# 4. stable identifier: HydroBASINS id + part counter (a basin may survive as >1 polygon)
gdf_catch["part"] = gdf_catch.groupby("hybas_id").cumcount()
gdf_catch["catchment_id"] = (
    gdf_catch["hybas_id"].astype("int64").astype(str) + "_" + gdf_catch["part"].astype(str)
)
gdf_catch = (gdf_catch.sort_values("catchment_id")
                      .drop(columns="part")
                      .reset_index(drop=True)
                      .to_crs(WORK_CRS))

dropped = len(gdf_exp) - len(gdf_catch)
print(f"exploded          : {len(gdf_exp)} single-part polygons")
print(f"kept              : {len(gdf_catch)}  (dropped {dropped} polygons < {MIN_CATCHMENT_AREA_KM2:g} km2, "
      f"{gdf_exp.loc[gdf_exp['area_km2'] < MIN_CATCHMENT_AREA_KM2, 'area_km2'].sum():.0f} km2 total)")
print(f"catchment area    : min {gdf_catch['area_km2'].min():.0f} | median "
      f"{gdf_catch['area_km2'].median():.0f} | max {gdf_catch['area_km2'].max():.0f} km2")
print(f"total area        : {gdf_catch['area_km2'].sum():,.0f} km2 "
      f"(Madagascar is ~587,000 km2)")

domain_gpkg = DIRS["domain"] / "pluvial_catchments_mdg.gpkg"
domain_geojson = DIRS["domain"] / "pluvial_catchments_mdg.geojson"
if OVERWRITE or not domain_gpkg.exists():
    gdf_catch.to_file(domain_gpkg, layer="pluvial_catchments", driver="GPKG")
    gdf_catch.to_file(domain_geojson, driver="GeoJSON")
register_output(domain_gpkg, dataset="pluvial catchments",
                source=f"HydroBASINS v1c level 07 ({HYBAS_FILE.name})",
                note=f"exploded, area >= {MIN_CATCHMENT_AREA_KM2:g} km2")
register_output(domain_geojson, dataset="pluvial catchments",
                source=f"HydroBASINS v1c level 07 ({HYBAS_FILE.name})", note="GeoJSON mirror")
gdf_catch[["catchment_id", "hybas_id", "area_km2", "up_area", "coast", "order"]].head()

### 2.1 Area-of-interest bounding box

Every static dataset in section 4 is clipped to the same AOI: the national bounding box of the
retained catchments plus `AOI_BUFFER_DEG`. Using one shared box guarantees that the DEM, land cover,
curve-number and soil grids all cover each catchment fully, including cells just outside the
boundary that the SFINCS subgrid routine needs.

In [ ]:
b = gdf_catch.total_bounds                                  # (minx, miny, maxx, maxy)
AOI_BOUNDS = (round(b[0] - AOI_BUFFER_DEG, 4), round(b[1] - AOI_BUFFER_DEG, 4),
              round(b[2] + AOI_BUFFER_DEG, 4), round(b[3] + AOI_BUFFER_DEG, 4))
AOI_GDF = gpd.GeoDataFrame(geometry=[box(*AOI_BOUNDS)], crs=WORK_CRS)

print(f"catchment bounds : {tuple(round(v, 4) for v in b)}")
print(f"AOI  (+{AOI_BUFFER_DEG} deg) : {AOI_BOUNDS}")
print(f"AOI  extent      : {AOI_BOUNDS[2] - AOI_BOUNDS[0]:.2f} deg lon x "
      f"{AOI_BOUNDS[3] - AOI_BOUNDS[1]:.2f} deg lat")
for name, res in (("FABDEM 30 m", 1 / 3600), ("VITO 100 m", 1 / 1008),
                  ("GCN250 250 m", 0.002083333), ("SoilGrids 250 m", 0.00225)):
    nx = (AOI_BOUNDS[2] - AOI_BOUNDS[0]) / res
    ny = (AOI_BOUNDS[3] - AOI_BOUNDS[1]) / res
    print(f"  {name:<16} {nx:>8.0f} x {ny:>8.0f} px  ({nx * ny / 1e6:>8.1f} Mpx)")

In [ ]:
if MAKE_FIGURES:
    fig, (ax_map, ax_hist) = plt.subplots(1, 2, figsize=(15, 8),
                                          gridspec_kw={"width_ratios": [1.25, 1]})

    # --- map of the retained catchments -------------------------------------
    gdf_catch.plot(ax=ax_map, column="area_km2", cmap="YlGnBu", legend=True, alpha=0.65,
                   edgecolor="0.25", linewidth=0.3, zorder=3,
                   legend_kwds={"label": "Catchment area (km$^2$)", "shrink": 0.7})
    AOI_GDF.boundary.plot(ax=ax_map, color="red", linewidth=1.4, linestyle="--", zorder=4)
    add_basemap(ax_map, source=cx.providers.Esri.WorldImagery if HAS_BASEMAP else None)
    ax_map.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)",
               title=f"Pluvial catchments ({len(gdf_catch)} SFINCS domains)")

    # --- area distribution, to justify the threshold ------------------------
    srt = gdf_exp["area_km2"].sort_values(ascending=False).values
    ax_hist.plot(np.arange(1, len(srt) + 1), srt, lw=1.5, color="#1f6f8b")
    ax_hist.axhline(MIN_CATCHMENT_AREA_KM2, color="red", ls="--",
                    label=f"threshold = {MIN_CATCHMENT_AREA_KM2:g} km$^2$")
    ax_hist.set(xscale="log", yscale="log", xlabel="Polygon rank (largest first)",
                ylabel="Area (km$^2$)", title="Area distribution of exploded polygons")
    ax_hist.legend(frameon=False)
    ax_hist.grid(which="both", color="0.9")
    ax_hist.set_axisbelow(True)

    plt.tight_layout()
    savefig("01_pluvial_catchments")
    plt.show()

---

# 3. Static datasets

Three grids feed the SFINCS pluvial schematisation directly (topography, land cover, curve number)
and one supports the infiltration parameterisation (soil texture / Hydrologic Soil Group).

All four are clipped to `AOI_BOUNDS`, kept on their **native grid and CRS (EPSG:4326)** — no
resampling here — and written as tiled, deflate-compressed GeoTIFFs with overviews so HydroMT can
read them efficiently at any zoom.

## 3.1 Topography — FABDEM V1-2 (30 m)

**FABDEM** (Forest And Buildings removed Copernicus DEM) is Copernicus GLO-30 with tree-canopy and
building height biases removed by a machine-learning correction. For pluvial modelling this matters
more than for fluvial: overland flow paths in vegetated and built-up areas are the ones that route
the rainfall, and the raw Copernicus surface model blocks them with canopy and roof artefacts.

* Grid: 1 arcsec (~30 m at the equator), EPSG:4326, `float32`, nodata `-9999`
* Vertical datum: **EGM2008 orthometric** — the same as Copernicus DEM. No datum shift is applied:
  pluvial modelling only needs relative elevations within a catchment, and SFINCS rain-on-grid has
  no sea-level boundary that would require an absolute datum. (Coupled pluvial-coastal runs *do*
  need one; convert to the local chart datum at that point, not here.)

### Access strategy

There is no tile-level REST API. The University of Bristol repository publishes the data as
**10° × 10° ZIP archives** (up to 861 MB each) containing 1° × 1° GeoTIFFs. Downloading whole
archives to keep a fraction of their contents is wasteful, so this notebook instead:

1. resolves which 1° tiles the AOI needs, and which archive holds each of them;
2. opens each remote archive through `HTTPRangeFile`, so `zipfile` reads only the **central
   directory** (a few kB at the end of the file);
3. extracts the needed members with HTTP `Range` requests — one ranged GET per tile.

The tiles are stored uncompressed inside the archives, so the bytes fetched equal the tile sizes:
**~1.4 GB for the 83 tiles covering Madagascar**, versus 1.47 GB of archives. At national scale the
saving in bytes is small, but the download becomes **per-tile, resumable and cacheable** — and for a
single-catchment run it fetches ~30 MB instead of 861 MB.

Naming convention (from the repository readme): tiles are labelled by their **south-west corner**
(`S19E047_FABDEM_V1-2.tif` spans 19–18 °S, 47–48 °E); archives are labelled
`<SW corner>-<NE corner>` (`S20E040-S10E050_FABDEM_V1-2.zip` spans 20–10 °S, 40–50 °E).

> `pip install fabdem` offers the same download as a one-liner, but it pulls whole archives. Use it
> if you prefer a maintained dependency over the ~40 lines below.

In [ ]:
FABDEM_BASE = "https://data.bris.ac.uk/datasets/s5hqmjcdj8yo2ibzi9b4ew3sn"
FABDEM_SUFFIX = "_FABDEM_V1-2"
FABDEM_NODATA = -9999.0


def _corner_label(lat: int, lon: int) -> str:
    """FABDEM corner label, e.g. (-19, 47) -> 'S19E047'."""
    return f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}{'E' if lon >= 0 else 'W'}{abs(lon):03d}"


def fabdem_tile_name(lat: int, lon: int) -> str:
    """1-degree tile file name from its south-west corner."""
    return f"{_corner_label(lat, lon)}{FABDEM_SUFFIX}.tif"


def fabdem_archive_name(lat: int, lon: int) -> str:
    """Name of the 10x10 degree archive holding the tile with that south-west corner."""
    a_lat, a_lon = math.floor(lat / 10) * 10, math.floor(lon / 10) * 10
    return (f"{_corner_label(a_lat, a_lon)}-{_corner_label(a_lat + 10, a_lon + 10)}"
            f"{FABDEM_SUFFIX}.zip")


def fabdem_tiles_for_bounds(bounds) -> List[Tuple[str, str]]:
    """[(tile_name, archive_name)] for every 1-degree cell intersecting *bounds* (W, S, E, N).

    The upper edges use ceil so a bound falling exactly on a degree line does not pull an extra row.
    """
    west, south, east, north = bounds
    lats = range(math.floor(south), math.ceil(north))
    lons = range(math.floor(west), math.ceil(east))
    return [(fabdem_tile_name(la, lo), fabdem_archive_name(la, lo)) for la in lats for lo in lons]


def download_fabdem(bounds, out_dir: Path, *, overwrite: bool = False) -> List[Path]:
    """Fetch the FABDEM tiles covering *bounds* into *out_dir*, one ranged GET per tile.

    Ocean-only cells are absent from the archives; those are reported and skipped, which is
    expected for an island AOI.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    wanted = fabdem_tiles_for_bounds(bounds)
    by_archive = {}   # archive -> [tile names]
    for tile, archive in wanted:
        by_archive.setdefault(archive, []).append(tile)

    paths, absent, fetched_bytes = [], [], 0
    print(f"{len(wanted)} candidate 1-degree cells in {len(by_archive)} archive(s)")

    for archive, tiles in sorted(by_archive.items()):
        todo = [t for t in tiles if overwrite or not (out_dir / t).exists()]
        cached = [t for t in tiles if not overwrite and (out_dir / t).exists()]
        paths += [out_dir / t for t in cached]
        if not todo:
            print(f"  {archive}: all {len(cached)} local tiles cached")
            continue
        url = f"{FABDEM_BASE}/{archive}"
        try:
            zf, raw = remote_zip(url)
        except requests.HTTPError as exc:            # archive does not exist = all-ocean block
            print(f"  {archive}: unavailable ({exc.response.status_code}), "
                  f"{len(todo)} cell(s) skipped")
            absent += todo
            continue
        members = set(zf.namelist())
        present = [t for t in todo if t in members]
        absent += [t for t in todo if t not in members]
        print(f"  {archive}: {len(members)} land tiles in archive, fetching {len(present)}"
              f"{f' ({len(cached)} cached)' if cached else ''}")
        for i, tile in enumerate(present, start=1):
            dest = out_dir / tile
            tmp = dest.with_suffix(".tif.part")
            with open(tmp, "wb") as fh:
                fh.write(zf.read(tile))
            tmp.replace(dest)
            fetched_bytes += dest.stat().st_size
            paths.append(dest)
            print(f"\r    {i}/{len(present)}  {tile}  ({human(dest.stat().st_size)})",
                  end="" if i < len(present) else "\n")
        zf.close()

    print(f"\ntiles on disk : {len(paths)}  ({human(sum(p.stat().st_size for p in paths))})")
    print(f"downloaded    : {human(fetched_bytes)} this run")
    if absent:
        print(f"absent (ocean): {len(absent)} cell(s), e.g. {[a[:7] for a in absent[:6]]}")
    return sorted(paths)


t0 = time.time()
fabdem_tiles = download_fabdem(AOI_BOUNDS, DIRS["fabdem_raw"], overwrite=OVERWRITE)
print(f"elapsed: {time.time() - t0:.0f}s")

### 3.1.1 Mosaic as a VRT

The 83 tiles are **not** merged into a single GeoTIFF: at 1 arcsec the national extent is
~28 800 × 54 000 px, i.e. ~6 GB of `float32`, most of it ocean nodata. Instead a **GDAL VRT** stitches
the tiles virtually — a small XML file that GDAL, rasterio and HydroMT all read as one raster, with
windowed reads pulling only the tiles they touch. Source paths are written **relative** to the VRT,
so `Data/` can be moved or shared without breaking it.

The XML is written directly rather than through `gdalbuildvrt` or `osgeo.gdal.BuildVRT`, so the
notebook needs neither the GDAL command-line tools nor the compiled Python bindings (see *Software
environments*). The format is simple for this case — every FABDEM tile shares one CRS, one
resolution and one dtype — and the cell **asserts** those three invariants rather than assuming
them. The mosaic is then **verified against every source tile pixel for pixel**, which is what makes
a hand-written VRT safe to rely on.

This VRT is the topography entry point in the SFINCS data catalogue (section 5.1).

In [ ]:
GDAL_DTYPE = {"uint8": "Byte", "int8": "Int8", "int16": "Int16", "uint16": "UInt16",
              "int32": "Int32", "uint32": "UInt32", "float32": "Float32", "float64": "Float64"}


def build_vrt(tiles: Sequence[Path], out_path: Path, nodata: float, crs: str = WORK_CRS) -> Path:
    """Write a VRT mosaicking *tiles*, with source paths relative to *out_path*.

    Requires every tile to share one CRS, one resolution and one dtype - true for a FABDEM
    tile set, and asserted here so a mixed input fails loudly instead of producing a
    silently misaligned mosaic.
    """
    info = []
    for tile in tiles:
        with rasterio.open(tile) as src:
            info.append(dict(path=Path(tile), width=src.width, height=src.height,
                             transform=src.transform, dtype=src.dtypes[0],
                             block=src.block_shapes[0], bounds=src.bounds, crs=src.crs))
    if not info:
        raise ValueError("no tiles to mosaic")

    xres, yres = info[0]["transform"].a, -info[0]["transform"].e
    for m in info:
        if not (math.isclose(m["transform"].a, xres, rel_tol=1e-9)
                and math.isclose(-m["transform"].e, yres, rel_tol=1e-9)):
            raise ValueError(f"{m['path'].name} resolution {m['transform'].a} != {xres}")
    dtypes = {m["dtype"] for m in info}
    if len(dtypes) > 1:
        raise ValueError(f"tiles have mixed dtypes: {sorted(dtypes)}")

    west = min(m["bounds"].left for m in info)
    east = max(m["bounds"].right for m in info)
    south = min(m["bounds"].bottom for m in info)
    north = max(m["bounds"].top for m in info)
    width = int(round((east - west) / xres))
    height = int(round((north - south) / yres))
    gdal_type = GDAL_DTYPE[info[0]["dtype"]]

    xml = [f'<VRTDataset rasterXSize="{width}" rasterYSize="{height}">',
           f'  <SRS>{crs}</SRS>',
           f'  <GeoTransform>{west!r}, {xres!r}, 0.0, {north!r}, 0.0, {-yres!r}</GeoTransform>',
           f'  <VRTRasterBand dataType="{gdal_type}" band="1">',
           f'    <NoDataValue>{nodata!r}</NoDataValue>',
           '    <ColorInterp>Gray</ColorInterp>']
    for m in info:
        rel = os.path.relpath(m["path"], out_path.parent)
        xoff = int(round((m["bounds"].left - west) / xres))
        yoff = int(round((north - m["bounds"].top) / yres))
        xml += [
            '    <ComplexSource>',
            f'      <SourceFilename relativeToVRT="1">{escape(rel)}</SourceFilename>',
            '      <SourceBand>1</SourceBand>',
            f'      <SourceProperties RasterXSize="{m["width"]}" RasterYSize="{m["height"]}"'
            f' DataType="{gdal_type}" BlockXSize="{m["block"][1]}" BlockYSize="{m["block"][0]}"/>',
            f'      <SrcRect xOff="0" yOff="0" xSize="{m["width"]}" ySize="{m["height"]}"/>',
            f'      <DstRect xOff="{xoff}" yOff="{yoff}" xSize="{m["width"]}" ySize="{m["height"]}"/>',
            f'      <NODATA>{nodata!r}</NODATA>',
            '    </ComplexSource>']
    xml += ['  </VRTRasterBand>', '</VRTDataset>', '']
    out_path.write_text("\n".join(xml))
    return out_path


def verify_vrt(vrt_path: Path, tiles: Sequence[Path]) -> None:
    """Assert the VRT returns each source tile's pixels exactly, at the right offset."""
    with rasterio.open(vrt_path) as mosaic:
        for tile in tiles:
            with rasterio.open(tile) as src:
                w = from_bounds(*src.bounds, transform=mosaic.transform)
                win = Window(int(round(w.col_off)), int(round(w.row_off)), src.width, src.height)
                if not np.array_equal(mosaic.read(1, window=win), src.read(1)):
                    raise AssertionError(f"VRT does not reproduce {tile.name} - misaligned mosaic")


dem_vrt = DIRS["topography"] / "fabdem_mdg.vrt"

if OVERWRITE or not dem_vrt.exists():
    build_vrt(fabdem_tiles, dem_vrt, FABDEM_NODATA)
    t0 = time.time()
    verify_vrt(dem_vrt, fabdem_tiles)
    print(f"VRT verified against all {len(fabdem_tiles)} source tiles, pixel for pixel "
          f"({time.time() - t0:.0f}s)")

register_output(dem_vrt, dataset="topography (FABDEM V1-2)", source=f"{FABDEM_BASE}/*.zip",
                note=f"virtual mosaic of {len(fabdem_tiles)} 1-degree tiles, EGM2008, CC BY-NC-SA 4.0")
describe_raster(dem_vrt, label="FABDEM V1-2 national mosaic (VRT)")

### 3.1.2 Coarse overview (QC only)

A ~250 m national mosaic, built by area-averaging the VRT through `rasterio.vrt.WarpedVRT`
(same operation as `gdalwarp -r average`, without the external binary). It exists so the DEM can be
plotted and sanity-checked in seconds; **it is not a model input** — SFINCS reads the
full-resolution VRT. Set `MAKE_DEM_OVERVIEW = False` to skip the one-off 3–8 minute build.

In [ ]:
dem_overview = DIRS["topography"] / f"fabdem_mdg_overview_{DEM_OVERVIEW_RES_DEG:g}deg.tif"

if MAKE_DEM_OVERVIEW and (OVERWRITE or not dem_overview.exists()):
    t0 = time.time()
    print(f"building {dem_overview.name} (averaging {len(fabdem_tiles)} tiles, this takes minutes)...")
    ov_width = int(math.ceil((AOI_BOUNDS[2] - AOI_BOUNDS[0]) / DEM_OVERVIEW_RES_DEG))
    ov_height = int(math.ceil((AOI_BOUNDS[3] - AOI_BOUNDS[1]) / DEM_OVERVIEW_RES_DEG))
    ov_transform = from_origin(AOI_BOUNDS[0], AOI_BOUNDS[3],
                               DEM_OVERVIEW_RES_DEG, DEM_OVERVIEW_RES_DEG)
    with rasterio.open(dem_vrt) as src:
        with WarpedVRT(src, crs=WORK_CRS, transform=ov_transform,
                       width=ov_width, height=ov_height, resampling=Resampling.average,
                       src_nodata=FABDEM_NODATA, nodata=FABDEM_NODATA) as wvrt:
            overview = wvrt.read(1)
    write_raster(dem_overview, overview, ov_transform, nodata=FABDEM_NODATA,
                 dtype="float32", descriptions=["elevation_m_egm2008"])
    print(f"  done in {time.time() - t0:.0f}s  ({ov_width} x {ov_height} px)")

if dem_overview.exists():
    register_output(dem_overview, dataset="topography overview",
                    source="derived from fabdem_mdg.vrt",
                    note=f"{DEM_OVERVIEW_RES_DEG} deg average resampling, QC only - not a model input")
    describe_raster(dem_overview, label="FABDEM overview")

In [ ]:
if MAKE_FIGURES and dem_overview.exists():
    with rasterio.open(dem_overview) as src:
        dem = src.read(1, masked=True)
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        px_deg = src.res[0]

    dem = np.ma.masked_less_equal(dem, -100)          # drop nodata and unrealistic values

    # simple hillshade for relief, elevation for the colour
    dy, dx = np.gradient(dem.filled(0).astype("float32"), 111_320 * px_deg)
    slope = np.arctan(np.hypot(dx, dy))
    aspect = np.arctan2(-dx, dy)
    az, alt = np.radians(315.0), np.radians(45.0)
    shade = (np.sin(alt) * np.cos(slope)
             + np.cos(alt) * np.sin(slope) * np.cos(az - aspect))

    fig, axes = plt.subplots(1, 2, figsize=(15, 9))
    im = axes[0].imshow(dem, extent=extent, cmap="terrain", vmin=0,
                        vmax=np.nanpercentile(dem.compressed(), 99.5))
    plt.colorbar(im, ax=axes[0], shrink=0.75, label="Elevation (m, EGM2008)")
    axes[0].set_title("FABDEM V1-2 elevation")

    axes[1].imshow(np.ma.masked_array(shade, dem.mask), extent=extent, cmap="gray",
                   vmin=0, vmax=1)
    axes[1].set_title("Hillshade (sun at 315° azimuth, 45° altitude)")

    for ax in axes:
        gdf_catch.boundary.plot(ax=ax, color="red", linewidth=0.35, alpha=0.8)
        ax.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)")
    plt.tight_layout()
    savefig("02_topography_fabdem")
    plt.show()

## 3.2 Land use / land cover — Copernicus Global Land Cover (VITO, 100 m)

The **Copernicus Global Land Service** discrete classification (PROBA-V LC100, collection 3,
epoch 2019) at 100 m. Two downstream uses:

* **Manning's *n***, via the class → roughness table below (applied during the SFINCS build);
* **infiltration**, as the land-cover half of the curve-number look-up (GCN250, section 3.3,
  already embeds this land cover, so the table here is the audit trail).

### Access strategy

Zenodo serves the global classification map as a **tiled GeoTIFF that honours HTTP range requests**,
so GDAL's `/vsicurl` driver reads just the AOI window out of the 1.7 GB global file — no full
download, no intermediate copy. The read below transfers ~100 MB and takes about a minute.

The class → Manning's *n* table is Deltares' `vito_mapping.csv` from HydroMT-SFINCS; it is saved
next to the raster so the roughness parameterisation is reproducible from this folder alone.

In [ ]:
VITO_URL = ("https://zenodo.org/records/3939050/files/"
            "PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326.tif")
VITO_MAPPING_URL = ("https://raw.githubusercontent.com/Deltares/hydromt_sfincs/main/"
                    "hydromt_sfincs/data/lulc/vito_mapping.csv")
VITO_NODATA = 255

lulc_tif = DIRS["lulc"] / "vito_lc100_2019_mdg.tif"
lulc_csv = DIRS["lulc"] / "vito_lc100_mapping.csv"

if OVERWRITE or not lulc_tif.exists():
    t0 = time.time()
    print(f"reading the AOI window out of the global VITO COG via /vsicurl ...")
    # GDAL_HTTP_USERAGENT is not optional: Zenodo answers 403 to GDAL's default user agent
    # (seen with GDAL 3.5; GDAL 3.10 is let through). Overriding it fixes the read on both.
    with rasterio.Env(GDAL_HTTP_USERAGENT="mdg-pluvial-preprocessing/1.0",
                      GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
                      GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=3):
        lulc, lulc_transform, lulc_meta = clip_remote_raster(f"/vsicurl/{VITO_URL}", AOI_BOUNDS)
    print(f"  window {lulc.shape[1]} x {lulc.shape[0]} px read in {time.time() - t0:.0f}s "
          f"(source grid {lulc_meta['source_shape'][1]} x {lulc_meta['source_shape'][0]})")
    write_raster(lulc_tif, lulc, lulc_transform, nodata=VITO_NODATA, dtype="uint8",
                 descriptions=["discrete_classification"])
else:
    with rasterio.open(lulc_tif) as src:
        lulc, lulc_transform = src.read(1), src.transform

if OVERWRITE or not lulc_csv.exists():
    pd.read_csv(VITO_MAPPING_URL).to_csv(lulc_csv, index=False)

register_output(lulc_tif, dataset="land cover (Copernicus VITO LC100 v3.0.1, 2019)",
                source=VITO_URL, note="AOI window, native 100 m grid, EPSG:4326")
register_output(lulc_csv, dataset="land cover look-up", source=VITO_MAPPING_URL,
                note="class -> description and Manning's n (Deltares hydromt_sfincs)")
describe_raster(lulc_tif, label="Copernicus VITO LC100 2019")

In [ ]:
# Class inventory over the AOI, joined to the Manning's n look-up
lulc_map = pd.read_csv(lulc_csv)
values, counts = np.unique(lulc, return_counts=True)
area_km2 = counts * (111.32 ** 2) * abs(lulc_transform.a * lulc_transform.e)   # deg^2 -> km^2, approx

inventory = (pd.DataFrame({"landuse": values, "pixels": counts, "area_km2": area_km2})
             .merge(lulc_map[["landuse", "description", "N"]], on="landuse", how="left")
             .assign(share_pct=lambda d: 100 * d["pixels"] / d["pixels"].sum())
             .sort_values("pixels", ascending=False)
             .reset_index(drop=True))
inventory["description"] = inventory["description"].fillna("(unmapped class)")
inventory = inventory.rename(columns={"N": "manning_n"})

land = inventory[~inventory["landuse"].isin([0, 200, VITO_NODATA])]
print(f"classes present : {len(inventory)}  ({len(land)} terrestrial)")
print(f"land area       : {land['area_km2'].sum():,.0f} km2 "
      f"({100 * land['pixels'].sum() / inventory['pixels'].sum():.1f}% of the AOI window)")
print(f"area-weighted Manning's n over land : "
      f"{np.average(land['manning_n'], weights=land['pixels']):.4f} s m^-1/3")
inventory[["landuse", "description", "manning_n", "pixels", "area_km2", "share_pct"]].round(
    {"area_km2": 0, "share_pct": 2})

In [ ]:
if MAKE_FIGURES:
    # official Copernicus CGLS-LC100 colours for the classes present
    VITO_COLORS = {
        0: "#282828", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000",
        60: "#b4b4b4", 70: "#f0f0f0", 80: "#0032c8", 90: "#0096a0", 100: "#fae6a0",
        111: "#58481f", 112: "#009900", 113: "#70663e", 114: "#00cc00", 115: "#4e751f",
        116: "#007800", 121: "#666000", 122: "#8db400", 123: "#8d7400", 124: "#a0dc00",
        125: "#929900", 126: "#648c00", 200: "#000080",
    }
    present = [v for v in inventory["landuse"] if v != VITO_NODATA]
    codes = np.full(int(max(present)) + 1, -1, dtype="int16")
    for i, v in enumerate(present):
        codes[int(v)] = i
    idx = np.where(lulc == VITO_NODATA, -1, codes[np.clip(lulc, 0, len(codes) - 1)])

    cmap = ListedColormap([VITO_COLORS.get(int(v), "#cccccc") for v in present])
    cmap.set_bad("white")

    fig, (ax, ax_bar) = plt.subplots(1, 2, figsize=(16, 9),
                                     gridspec_kw={"width_ratios": [1.15, 1]})
    ax.imshow(np.ma.masked_less(idx, 0),
              extent=(AOI_BOUNDS[0], AOI_BOUNDS[2], AOI_BOUNDS[1], AOI_BOUNDS[3]),
              cmap=cmap, vmin=-0.5, vmax=len(present) - 0.5, interpolation="nearest")
    gdf_catch.boundary.plot(ax=ax, color="0.2", linewidth=0.3, alpha=0.6)
    ax.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)",
           title="Copernicus VITO LC100 (2019), 100 m")

    top = land.head(12).iloc[::-1]
    ax_bar.barh(top["description"].str.slice(0, 42), top["area_km2"],
                color=[VITO_COLORS.get(int(v), "#cccccc") for v in top["landuse"]],
                edgecolor="0.3")
    for y, (a, n) in enumerate(zip(top["area_km2"], top["manning_n"])):
        ax_bar.text(a, y, f"  n={n:.3f}", va="center", fontsize=10, color="0.3")
    ax_bar.set(xlabel="Area (km$^2$)", title="Terrestrial classes by area, with Manning's $n$")
    ax_bar.grid(axis="x", color="0.9")
    ax_bar.set_axisbelow(True)
    for side in ("top", "right"):
        ax_bar.spines[side].set_visible(False)

    plt.tight_layout()
    savefig("03_landcover_vito")
    plt.show()

## 3.3 Curve Number — GCN250 (250 m)

**GCN250** (Jaafar, Ahmad & El Beyrouthy 2019) is a global SCS Curve Number grid built by crossing
ESA CCI land cover with HYSOGs250m hydrologic soil groups on the standard NRCS TR-55 table. It gives
SFINCS the infiltration/effective-rainfall term of the rain-on-grid computation without having to
rebuild the CN cross-tabulation locally.

Three grids, one per **antecedent runoff condition (ARC)**:

| Layer | ARC | Antecedent soil moisture | Use in the pluvial model |
|-------|-----|--------------------------|--------------------------|
| `GCN250_ARCI` | I | dry | lower bound / sensitivity |
| `GCN250_ARCII` | II | average | **baseline** |
| `GCN250_ARCIII` | III | wet | upper bound; realistic for a cyclone arriving on saturated soils |

`GCN250_ARCS` in the configuration controls which are fetched. Start with ARC II; ARC III matters
for Madagascar because the damaging events are cyclones landing in an already-wet season, so a
paired ARC II / ARC III run is the natural sensitivity test.

### Access strategy

The dataset lives on figshare (doi:10.6084/m9.figshare.7756202). The **figshare REST API** is queried
for the current download URLs rather than hard-coding them — figshare mints time-limited
pre-signed S3 links (≈10 s validity), so `/vsicurl` streaming is not an option and each global
GeoTIFF (~640 MB) is cached locally once, then windowed.

Values are `uint8` CN (0–100) with `255` as nodata over the ocean.

In [ ]:
GCN250_FIGSHARE_ID = 7756202
GCN250_NODATA = 255


def figshare_files(article_id: int) -> Dict[str, dict]:
    """{file name: metadata} for a figshare article, via the public REST API."""
    resp = SESSION.get(f"https://api.figshare.com/v2/articles/{article_id}", timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    meta = resp.json()
    print(f"figshare {article_id}: {meta['title']}  (doi {meta['doi']})")
    return {f["name"]: f for f in meta["files"]}


gcn_files = figshare_files(GCN250_FIGSHARE_ID)
for name, meta in gcn_files.items():
    print(f"  {name:<22} {meta['size'] / 1e6:8.1f} MB")

gcn_local = {}
for arc in GCN250_ARCS:
    fname = f"GCN250_{arc}.tif"
    if fname not in gcn_files:
        raise KeyError(f"{fname} not published in article {GCN250_FIGSHARE_ID}: "
                       f"{sorted(gcn_files)}")
    gcn_local[arc] = http_download(gcn_files[fname]["download_url"],
                                   DIRS["gcn250_raw"] / fname, overwrite=OVERWRITE)

In [ ]:
gcn_clipped = {}
for arc, src_path in gcn_local.items():
    out = DIRS["curve_number"] / f"gcn250_{arc.lower()}_mdg.tif"
    if OVERWRITE or not out.exists():
        cn, transform, meta = clip_remote_raster(str(src_path), AOI_BOUNDS)
        # ocean and unclassified cells -> nodata, so CN statistics are land-only
        cn = np.where((cn == 0) | (cn > 100), GCN250_NODATA, cn).astype("uint8")
        write_raster(out, cn, transform, nodata=GCN250_NODATA, dtype="uint8",
                     descriptions=[f"curve_number_{arc}"])
    gcn_clipped[arc] = register_output(
        out, dataset=f"curve number GCN250 {arc}",
        source=f"figshare:{GCN250_FIGSHARE_ID}/GCN250_{arc}.tif",
        note="AOI window, native 250 m grid; 0 and >100 recoded to nodata 255")
    describe_raster(out, label=f"GCN250 {arc}")
    print()

# land-only CN statistics per ARC
rows = []
for arc, path in gcn_clipped.items():
    with rasterio.open(path) as src:
        a = src.read(1)
    v = a[a != GCN250_NODATA].astype("float32")
    rows.append({"ARC": arc, "valid_px": v.size,
                 "CN_min": v.min(), "CN_p10": np.percentile(v, 10),
                 "CN_median": np.median(v), "CN_mean": v.mean(),
                 "CN_p90": np.percentile(v, 90), "CN_max": v.max(),
                 "S_mm_at_mean_CN": 25.4 * (1000 / v.mean() - 10)})
pd.DataFrame(rows).round(1)

In [ ]:
if MAKE_FIGURES:
    arcs = list(gcn_clipped)
    fig, axes = plt.subplots(1, len(arcs) + 1, figsize=(6 * len(arcs) + 6, 8.5),
                             gridspec_kw={"width_ratios": [1] * len(arcs) + [0.9]})
    axes = np.atleast_1d(axes)
    levels = [30, 40, 50, 60, 70, 75, 80, 85, 90, 95, 100]
    cmap = plt.get_cmap("RdYlGn_r", len(levels) - 1)
    norm = mcolors.BoundaryNorm(levels, cmap.N)

    for ax, arc in zip(axes, arcs):
        with rasterio.open(gcn_clipped[arc]) as src:
            cn = src.read(1, masked=True)
            ext = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        cn = np.ma.masked_equal(cn, GCN250_NODATA)
        im = ax.imshow(cn, extent=ext, cmap=cmap, norm=norm, interpolation="nearest")
        gdf_catch.boundary.plot(ax=ax, color="0.25", linewidth=0.3, alpha=0.7)
        ax.set(xlabel="Longitude (deg)", title=f"GCN250 {arc}")
        ax.set_ylabel("Latitude (deg)" if arc == arcs[0] else "")
    plt.colorbar(im, ax=axes[len(arcs) - 1], shrink=0.7, ticks=levels,
                 label="Curve Number (-)")

    ax_h = axes[-1]
    for arc in arcs:
        with rasterio.open(gcn_clipped[arc]) as src:
            a = src.read(1)
        v = a[a != GCN250_NODATA]
        ax_h.hist(v, bins=np.arange(20, 102, 2), histtype="step", lw=2, density=True, label=arc)
    ax_h.set(xlabel="Curve Number (-)", ylabel="Density",
             title="CN distribution over land")
    ax_h.legend(frameon=False)
    ax_h.grid(color="0.9")
    ax_h.set_axisbelow(True)
    for side in ("top", "right"):
        ax_h.spines[side].set_visible(False)

    plt.tight_layout()
    savefig("04_curve_number_gcn250")
    plt.show()

## 3.4 Soil texture and Hydrologic Soil Group — SoilGrids 2.0 (250 m)

GCN250 already bundles soil information, but it is a fixed cross-tabulation. An explicit soil layer
is needed to (a) audit the CN field and (b) parameterise **Green-Ampt** infiltration, which SFINCS
supports as an alternative to SCS-CN and which needs hydraulic conductivity and suction — both
usually derived from texture.

Products written here:

| Output | Content |
|--------|---------|
| `soilgrids_sand_0-100cm_mdg.tif` | depth-weighted sand fraction (%) |
| `soilgrids_clay_0-100cm_mdg.tif` | depth-weighted clay fraction (%) |
| `usda_texture_mdg.tif` | USDA texture class, 1–12 |
| `hsg_mdg.tif` | Hydrologic Soil Group, 1–4 = A–D |

### Access strategy

ISRIC exposes SoilGrids through **WCS 2.0.1**. `GetCoverage` accepts the subset *and* the output CRS,
so the server delivers the Madagascar window already reprojected from the native Homolosine grid to
EPSG:4326 — 10 requests (2 variables × 5 depths), ~8 MB and ~3 s each.

Values are returned as `int16` in g/kg; dividing by 10 gives percent. `0` marks ocean/no-data.

### From texture to Hydrologic Soil Group

1. Depth-weighted mean sand and clay over **0–100 cm**, weights = layer thickness
   (5, 10, 15, 30, 40 cm).
2. Silt = 100 − sand − clay; classify into the **12 USDA texture classes** using the standard
   texture-triangle rules.
3. Map texture → HSG with the NRCS texture shortcut (NEH-630 ch. 7):

   | HSG | Textures | Infiltration |
   |-----|----------|--------------|
   | A | sand, loamy sand, sandy loam | high |
   | B | loam, silt loam, silt | moderate |
   | C | sandy clay loam | slow |
   | D | clay loam, silty clay loam, sandy clay, silty clay, clay | very slow |

> **Caveat.** The texture shortcut ignores depth to bedrock and to the water table, both of which can
> demote a soil by one or two groups. Where that matters, replace this layer with **HYSOGs250m**
> (Ross et al. 2018, ORNL DAAC) — the dataset GCN250 itself uses. It is not fetched here because it
> is behind NASA Earthdata authentication; drop it into `processed/soil/` and point the catalogue at
> it if you have credentials.

In [ ]:
SOILGRIDS_WCS = "https://maps.isric.org/mapserv?map=/map/{var}.map"
SOILGRIDS_NODATA = 0


def soilgrids_coverage(var: str, depth: str, bounds, dest: Path, *,
                       stat: str = "mean", overwrite: bool = False) -> Path:
    """Download one SoilGrids coverage over *bounds*, reprojected to EPSG:4326 by the server."""
    if dest.exists() and not overwrite:
        return dest
    west, south, east, north = bounds
    epsg4326 = "http://www.opengis.net/def/crs/EPSG/0/4326"
    params = {
        "SERVICE": "WCS", "VERSION": "2.0.1", "REQUEST": "GetCoverage",
        "COVERAGEID": f"{var}_{depth}_{stat}", "FORMAT": "GEOTIFF_INT16",
        "SUBSETTINGCRS": epsg4326, "OUTPUTCRS": epsg4326,
    }
    url = SOILGRIDS_WCS.format(var=var)
    query = "&".join(f"{k}={v}" for k, v in params.items())
    query += f"&SUBSET=X({west},{east})&SUBSET=Y({south},{north})"
    resp = SESSION.get(f"{url}&{query}", timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    if not resp.content.startswith((b"II", b"MM")):        # WCS reports errors as XML, HTTP 200
        raise RuntimeError(f"WCS did not return a GeoTIFF for {var}_{depth}: "
                           f"{resp.content[:400].decode('utf-8', 'replace')}")
    dest.write_bytes(resp.content)
    return dest


sg_raw_dir = RAW_DIR / "soilgrids"
sg_raw_dir.mkdir(parents=True, exist_ok=True)

sg_layers = {"sand": {}, "clay": {}}          # variable -> depth -> file
t0 = time.time()
for var in sg_layers:
    for depth in SOILGRIDS_DEPTHS:
        dest = sg_raw_dir / f"soilgrids_{var}_{depth}_mean.tif"
        sg_layers[var][depth] = soilgrids_coverage(var, depth, AOI_BOUNDS, dest,
                                                   overwrite=OVERWRITE)
        print(f"  {var:<5} {depth:<8} {human(dest.stat().st_size)}")
print(f"{2 * len(SOILGRIDS_DEPTHS)} coverages in {time.time() - t0:.0f}s")

with rasterio.open(sg_layers["sand"][SOILGRIDS_DEPTHS[0]]) as src:
    sg_transform, sg_shape, sg_crs = src.transform, (src.height, src.width), src.crs
print(f"SoilGrids grid : {sg_shape[1]} x {sg_shape[0]} px @ {src.res[0]:.6f} deg ({sg_crs})")

In [ ]:
def depth_weighted_mean(paths: Dict[str, Path], weights: Sequence[float]) -> np.ndarray:
    """Thickness-weighted mean over the SoilGrids depth intervals, in percent."""
    stack, w = [], []
    for (depth, path), weight in zip(paths.items(), weights):
        with rasterio.open(path) as src:
            layer = src.read(1).astype("float32")
        layer[layer == SOILGRIDS_NODATA] = np.nan
        stack.append(layer / 10.0)                      # g/kg -> %
        w.append(weight)
    stack = np.stack(stack)
    w = np.asarray(w, dtype="float32")[:, None, None]
    valid = np.isfinite(stack)
    with np.errstate(invalid="ignore", divide="ignore"):     # all-nodata cells divide by 0
        mean = (np.nansum(np.where(valid, stack * w, 0), axis=0)
                / np.sum(w * valid, axis=0))
    return np.where(valid.any(axis=0), mean, np.nan)


sand = depth_weighted_mean(sg_layers["sand"], SOILGRIDS_WEIGHTS)
clay = depth_weighted_mean(sg_layers["clay"], SOILGRIDS_WEIGHTS)
silt = 100.0 - sand - clay

USDA_CLASSES = ["sand", "loamy sand", "sandy loam", "loam", "silt loam", "silt",
                "sandy clay loam", "clay loam", "silty clay loam", "sandy clay",
                "silty clay", "clay"]
TEXTURE_TO_HSG = {  # NRCS NEH-630 ch. 7 texture shortcut
    "sand": 1, "loamy sand": 1, "sandy loam": 1,
    "loam": 2, "silt loam": 2, "silt": 2,
    "sandy clay loam": 3,
    "clay loam": 4, "silty clay loam": 4, "sandy clay": 4, "silty clay": 4, "clay": 4,
}


def usda_texture(sand: np.ndarray, clay: np.ndarray, silt: np.ndarray) -> np.ndarray:
    """USDA texture class (1-12, 0 = nodata) from sand/clay/silt percentages.

    Conditions are evaluated in the canonical order of the USDA texture triangle, so the
    first match wins and the class boundaries need no special-casing.
    """
    with np.errstate(invalid="ignore"):
        conditions = [
            silt + 1.5 * clay < 15,                                                   # sand
            (silt + 1.5 * clay >= 15) & (silt + 2 * clay < 30),                       # loamy sand
            ((clay >= 7) & (clay < 20) & (sand > 52) & (silt + 2 * clay >= 30))
            | ((clay < 7) & (silt < 50) & (sand > 43)),                               # sandy loam
            (clay >= 7) & (clay < 27) & (silt >= 28) & (silt < 50) & (sand <= 52),     # loam
            ((silt >= 50) & (clay >= 12) & (clay < 27))
            | ((silt >= 50) & (silt < 80) & (clay < 12)),                              # silt loam
            (silt >= 80) & (clay < 12),                                                # silt
            (clay >= 20) & (clay < 35) & (silt < 28) & (sand > 45),                    # sandy clay loam
            (clay >= 27) & (clay < 40) & (sand > 20) & (sand <= 45),                   # clay loam
            (clay >= 27) & (clay < 40) & (sand <= 20),                                 # silty clay loam
            (clay >= 35) & (sand > 45),                                                # sandy clay
            (clay >= 40) & (silt >= 40),                                               # silty clay
            (clay >= 40) & (sand <= 45) & (silt < 40),                                 # clay
        ]
    texture = np.select(conditions, np.arange(1, 13), default=0).astype("uint8")
    return np.where(np.isfinite(sand) & np.isfinite(clay), texture, 0).astype("uint8")


texture = usda_texture(sand, clay, silt)
hsg = np.zeros_like(texture)
for i, name in enumerate(USDA_CLASSES, start=1):
    hsg[texture == i] = TEXTURE_TO_HSG[name]

soil_out = {}
for name, arr, nodata, dtype, desc in [
    ("soilgrids_sand_0-100cm_mdg.tif", sand, -9999.0, "float32", "sand_pct_0-100cm"),
    ("soilgrids_clay_0-100cm_mdg.tif", clay, -9999.0, "float32", "clay_pct_0-100cm"),
    ("usda_texture_mdg.tif", texture, 0, "uint8", "usda_texture_class_1-12"),
    ("hsg_mdg.tif", hsg, 0, "uint8", "hydrologic_soil_group_1A_2B_3C_4D"),
]:
    path = DIRS["soil"] / name
    if OVERWRITE or not path.exists():
        write_raster(path, np.where(np.isfinite(arr), arr, nodata) if dtype == "float32" else arr,
                     sg_transform, nodata=nodata, dtype=dtype, descriptions=[desc])
    soil_out[name] = register_output(
        path, dataset="soil (SoilGrids 2.0)",
        source="https://maps.isric.org WCS 2.0.1, sand/clay mean, 0-100 cm",
        note=desc + f"; depth weights {SOILGRIDS_WEIGHTS} cm")

for name in soil_out:
    describe_raster(soil_out[name], label=name)
    print()

# composition table
tex_tab = (pd.DataFrame({"class": np.arange(1, 13), "texture": USDA_CLASSES})
           .assign(hsg=lambda d: d["texture"].map(TEXTURE_TO_HSG).map({1: "A", 2: "B", 3: "C", 4: "D"}),
                   pixels=lambda d: [int((texture == c).sum()) for c in d["class"]]))
tex_tab["share_pct"] = (100 * tex_tab["pixels"] / max(int((texture > 0).sum()), 1)).round(2)
print(f"soil pixels classified : {int((texture > 0).sum()):,} "
      f"({100 * (texture > 0).sum() / texture.size:.1f}% of the AOI window)")
print("HSG shares (% of soil pixels):",
      {g: round(100 * (hsg == i).sum() / max((hsg > 0).sum(), 1), 1)
       for i, g in enumerate("ABCD", start=1)})
tex_tab.sort_values("pixels", ascending=False).reset_index(drop=True)

In [ ]:
if MAKE_FIGURES:
    fig, axes = plt.subplots(1, 3, figsize=(19, 8))
    ext = (AOI_BOUNDS[0], AOI_BOUNDS[2], AOI_BOUNDS[1], AOI_BOUNDS[3])

    for ax, arr, title, cmap in [
        (axes[0], np.ma.masked_invalid(sand), "Sand fraction, 0-100 cm (%)", "YlOrBr"),
        (axes[1], np.ma.masked_invalid(clay), "Clay fraction, 0-100 cm (%)", "BuPu"),
    ]:
        im = ax.imshow(arr, extent=ext, cmap=cmap, vmin=0, vmax=80, interpolation="nearest")
        plt.colorbar(im, ax=ax, shrink=0.7, label="%")
        ax.set(xlabel="Longitude (deg)", title=title)

    hsg_cmap = ListedColormap(["#2c7fb8", "#7fcdbb", "#fec44f", "#d95f0e"])
    im = axes[2].imshow(np.ma.masked_equal(hsg, 0), extent=ext, cmap=hsg_cmap,
                        vmin=0.5, vmax=4.5, interpolation="nearest")
    cbar = plt.colorbar(im, ax=axes[2], shrink=0.7, ticks=[1, 2, 3, 4])
    cbar.ax.set_yticklabels(["A (high)", "B (moderate)", "C (slow)", "D (very slow)"])
    cbar.set_label("Hydrologic Soil Group")
    axes[2].set(xlabel="Longitude (deg)", title="HSG from USDA texture")

    for ax in axes:
        gdf_catch.boundary.plot(ax=ax, color="0.2", linewidth=0.3, alpha=0.6)
    axes[0].set_ylabel("Latitude (deg)")
    plt.tight_layout()
    savefig("05_soil_hsg")
    plt.show()

---

# 4. Dynamic dataset — rainfall

Rain-on-grid forcing comes from **GPM IMERG Final Run daily** precipitation (0.1°, 1998-01-01 →
2024-09-30), pre-clipped to the Madagascar window in `Data/gpm_daily_precipitation_pp.nc`.

Two forcing families are prepared:

1. **Observed events** — the rainfall field of a specific historical cyclone, exported as a SFINCS
   `netamprfile`. Used for calibration and validation against reported flooding.
2. **Design hyetographs** — a return-period rainfall depth disaggregated to a synthetic 30-minute
   sequence. Used for the hazard maps that go into the risk layer.

The catalogue of cyclones and the TC attribution below serve both: they identify which historical
events are worth simulating, and they quantify how much of Madagascar's rainfall is cyclone-driven,
which is the argument for building the design events around cyclone rainfall rather than annual
maxima of all-cause rainfall.

> **Temporal resolution caveat.** The available cube is **daily**. SFINCS rain-on-grid needs
> sub-daily intensities to reproduce flash-flood peaks. Section 4.6 therefore disaggregates the daily
> depths into 30-minute blocks with a documented empirical relation. The proper fix is to ingest
> **IMERG Final Run half-hourly** (`GPM_3IMERGHH`, 0.1°, 30 min) for the selected event windows —
> only a few GB once the events are chosen. The export in section 4.5 writes whatever time step the
> input cube has, so it needs no change when the half-hourly data arrives.

## 4.1 Tropical cyclone catalogue

Storms known to have made landfall on, or passed close to, Madagascar between 2000 and 2022, with
the approximate date of passage. This list is the *query*: each entry is matched against IBTrACS
below to recover the full best track.

In [ ]:
storm_list = [
    ("ELINE", "2000-02-17"), ("GLORIA", "2000-03-01"), ("HUDAH", "2000-04-02"),
    ("CYPRIEN", "2002-01-02"), ("HARY", "2002-03-10"), ("KESINY", "2002-05-09"),
    ("ATANG", "2002-11-10"), ("FARI", "2003-01-28"), ("MANOU", "2003-05-08"),
    ("CELA", "2003-12-09"), ("ELITA", "2004-01-28"), ("GAFILO", "2004-03-07"),
    ("ERNEST", "2005-01-23"), ("FELAPI", "2005-01-28"), ("BOLOETSE", "2006-02-04"),
    ("BONDO", "2006-12-25"), ("CLOVIS", "2007-01-03"), ("FAVIO", "2007-02-19"),
    ("INDLALA", "2007-03-15"), ("JAYA", "2007-04-03"), ("FAME", "2008-01-27"),
    ("IVAN", "2008-02-17"), ("JOKWE", "2008-03-05"), ("ERIC", "2009-01-19"),
    ("FANELE", "2009-01-21"), ("GAEL", "2009-02-07"), ("IZILDA", "2009-03-26"),
    ("JADE", "2009-04-06"), ("FAMI", "2010-02-02"), ("HUBERT", "2010-03-10"),
    ("JOEL", "2010-05-27"), ("BINGIZA", "2011-02-14"), ("03_2011", "2011-12-19"),
    ("CHANDA", "2012-01-09"), ("GIOVANNA", "2012-02-13"), ("IRINA", "2012-02-26"),
    ("FELLENG", "2013-01-31"), ("HARUNA", "2013-02-22"), ("DELIWE", "2014-01-15"),
    ("HELLEN", "2014-03-31"), ("CHEDZA", "2015-01-16"), ("FUNDI", "2015-02-06"),
    ("ENAWO", "2017-03-07"), ("AVA", "2018-01-05"), ("DUMAZILE", "2018-03-03"),
    ("ELIAKIM", "2018-03-16"), ("EKETSANG", "2019-01-24"), ("IDAI", "2019-03-10"),
    ("BELNA", "2019-12-09"), ("DIANE", "2019-12-27"), ("FRANCISCO", "2020-02-15"),
    ("HEROLD", "2020-03-14"), ("CHALANE", "2020-12-26"), ("ELOISE", "2021-01-19"),
    ("IMAN", "2021-03-05"), ("ANA", "2022-01-22"), ("BATSIRAI", "2022-02-05"),
    ("DUMAKO", "2022-02-15"), ("EMNATI", "2022-02-22"), ("GOMBE", "2022-03-07"),
]
storm_df = pd.DataFrame(storm_list, columns=["name", "ref_date"])
storm_df["ref_date"] = pd.to_datetime(storm_df["ref_date"])

IBTRACS_YEAR_RANGE = (1999, 2023)       # 1 year of padding either side of the catalogue
IBTRACS_BASIN = "SI"                    # South Indian

print(f"{len(storm_df)} storms, {storm_df['ref_date'].dt.year.min()}-"
      f"{storm_df['ref_date'].dt.year.max()}")
storm_df.head()

## 4.2 Best-track retrieval and matching (IBTrACS v04r00)

The cyclone catalogue above is matched against **IBTrACS**, NOAA's consolidated best-track archive.
The **South Indian basin file is downloaded directly from NOAA NCEI** — 7 MB of netCDF holding every
SI storm since 1848. That is the whole dependency: no CLIMADA, no API client, no authentication.

Matching runs in three steps:

1. **Geographic screen.** Keep only storms with at least one fix within `MATCH_BUFFER_DEG` (8°) of
   the AOI. This is what stops a same-named or same-dated system from the far side of the basin
   being picked up — without it, the unnamed `03_2011` entry matches an Australian-basin storm
   4 000 km away.
2. **Name match**, case-insensitive. IBTrACS concatenates names for systems renamed across basins
   (`ELINE:LEONE`), so the target is tested against each colon-separated token.
3. **Date disambiguation.** Names are recycled across seasons, so among the candidates the one whose
   active window sits closest to the catalogue's date of passage wins. Entries with no usable name
   (`03_2011`) skip step 2 and fall back to a date-only match with a ±5 day tolerance.

Anything still unmatched is **reported, not silently substituted**, together with the nearest
candidates so it can be resolved by hand.

**Wind speed.** `usa_wind` (1-minute sustained, knots) is used where present — it is what the
Saffir-Simpson thresholds in section 4.2.1 are defined on. Where it is missing, `wmo_wind` is used
instead; in this basin that is La Réunion's 10-minute sustained wind, converted with the
conventional 10-min → 1-min factor of 1/0.88.

The result is flattened to one row per track fix and cached in
`processed/rainfall/tc_tracks_mdg.csv`. Later cells read that cache, so this cell only runs once —
delete the CSV to refresh from IBTrACS.

In [ ]:
IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-"
               "stewardship-ibtracs/v04r00/access/netcdf/IBTrACS.{basin}.v04r00.nc")
WMO_TO_1MIN = 1 / 0.88        # 10-min sustained (La Reunion) -> 1-min sustained
MATCH_BUFFER_DEG = 8.0        # a candidate must pass this close to the AOI
DATE_TOLERANCE_DAYS = 5       # window for the name-less fallback match

tracks_csv = DIRS["rainfall"] / "tc_tracks_mdg.csv"


def ibtracs_fixes(nc_path: Path, season_range=IBTRACS_YEAR_RANGE) -> pd.DataFrame:
    """Flatten an IBTrACS basin file to one row per track fix.

    Returns columns: sid, ibtracs_name, season, time, lat, lon, vmax_kn.
    """
    with xr.open_dataset(nc_path) as ib:
        def _text(var):
            return np.char.strip(np.char.decode(ib[var].values.astype("S"), "utf-8"))

        season = ib["season"].values
        keep = np.where((season >= season_range[0]) & (season <= season_range[1]))[0]
        n_fix = ib.sizes["date_time"]
        time = ib["time"].values[keep]
        lat, lon = ib["lat"].values[keep], ib["lon"].values[keep]
        usa, wmo = ib["usa_wind"].values[keep], ib["wmo_wind"].values[keep]
        sid, name = _text("sid")[keep], np.char.upper(_text("name")[keep])

    vmax = np.where(np.isfinite(usa), usa, wmo * WMO_TO_1MIN)
    valid = (~pd.isna(time.ravel())) & np.isfinite(lat.ravel()) & np.isfinite(lon.ravel())
    out = pd.DataFrame({
        "sid": np.repeat(sid, n_fix)[valid],
        "ibtracs_name": np.repeat(name, n_fix)[valid],
        "season": np.repeat(season[keep], n_fix)[valid].astype("int16"),
        "time": pd.to_datetime(time.ravel()[valid]).round("min"),   # IBTrACS times carry ns dust
        "lat": lat.ravel()[valid].astype("float64"),
        "lon": lon.ravel()[valid].astype("float64"),
        "vmax_kn": np.nan_to_num(vmax.ravel()[valid], nan=0.0).astype("float64"),
    })
    return out.sort_values(["sid", "time"], ignore_index=True)


def match_catalogue(fixes: pd.DataFrame, catalogue: pd.DataFrame, bounds,
                    buffer_deg=MATCH_BUFFER_DEG):
    """Match each catalogue entry to an IBTrACS storm id. Returns (tracks, report)."""
    west, south, east, north = bounds
    near = (fixes["lon"].between(west - buffer_deg, east + buffer_deg)
            & fixes["lat"].between(south - buffer_deg, north + buffer_deg))
    local = fixes[fixes["sid"].isin(fixes.loc[near, "sid"].unique())]
    print(f"IBTrACS  : {fixes['sid'].nunique()} storms in {IBTRACS_BASIN} "
          f"{IBTRACS_YEAR_RANGE}, {local['sid'].nunique()} passing within "
          f"{buffer_deg:g} deg of the AOI")

    span = local.groupby("sid").agg(name=("ibtracs_name", "first"),
                                    t0=("time", "min"), t1=("time", "max"))
    tokens = {sid: set(str(nm).split(":")) for sid, nm in span["name"].items()}

    rows, report = [], []
    for _, entry in catalogue.iterrows():
        target, ref = entry["name"], entry["ref_date"]
        named = not target[0].isdigit()
        cand = [sid for sid, tk in tokens.items() if target.upper() in tk] if named else []
        how = "name"
        if not cand:
            how = "date"
            tol = pd.Timedelta(days=DATE_TOLERANCE_DAYS)
            cand = [sid for sid in span.index
                    if span.at[sid, "t0"] - tol <= ref <= span.at[sid, "t1"] + tol]
        if not cand:
            nearest = (span.assign(gap=(span["t0"] - ref).abs())
                           .nsmallest(3, "gap")[["name", "t0", "t1"]])
            report.append({"name": target, "matched": False, "how": "-", "sid": None,
                           "note": "nearest local storms: " + "; ".join(
                               f"{r['name']} {r['t0'].date()}->{r['t1'].date()}"
                               for _, r in nearest.iterrows())})
            continue

        best = min(cand, key=lambda sid: (local.loc[local["sid"] == sid, "time"] - ref).abs().min())
        track = local[local["sid"] == best].copy()
        track["name"] = target                      # catalogue name is the downstream key
        rows.append(track)
        report.append({"name": target, "matched": True, "how": how, "sid": best,
                       "note": f"IBTrACS {span.at[best, 'name']}, "
                               f"{track['time'].min().date()} -> {track['time'].max().date()}"})

    tracks = (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame())
    return tracks, pd.DataFrame(report)


if tracks_csv.exists() and not OVERWRITE:
    tracks_df = pd.read_csv(tracks_csv, parse_dates=["time"])
    print(f"loaded cached best tracks: {tracks_csv.relative_to(EXEC_DIR)}")
else:
    ib_path = http_download(IBTRACS_URL.format(basin=IBTRACS_BASIN),
                            RAW_DIR / f"IBTrACS.{IBTRACS_BASIN}.v04r00.nc", overwrite=OVERWRITE)
    all_fixes = ibtracs_fixes(ib_path)
    tracks_df, match_report = match_catalogue(all_fixes, storm_df, AOI_BOUNDS)

    n_ok = int(match_report["matched"].sum())
    print(f"matched  : {n_ok} of {len(storm_df)} catalogue entries "
          f"({int((match_report['how'] == 'date').sum())} by date fallback)")
    if n_ok < len(match_report):
        print("\nUNMATCHED - resolve by hand, or drop from storm_list:")
        for _, r in match_report[~match_report["matched"]].iterrows():
            print(f"  {r['name']}: {r['note']}")

    tracks_df = tracks_df[["name", "sid", "season", "time", "lat", "lon", "vmax_kn"]]
    tracks_df.to_csv(tracks_csv, index=False)
    register_output(tracks_csv, dataset="TC best tracks",
                    source=IBTRACS_URL.format(basin=IBTRACS_BASIN),
                    note=f"{tracks_df['name'].nunique()} of {len(storm_df)} catalogue storms "
                         f"matched; vmax = usa_wind, else wmo_wind / 0.88")
    match_report.to_csv(DIRS["rainfall"] / "tc_track_match_report.csv", index=False)
    register_output(DIRS["rainfall"] / "tc_track_match_report.csv",
                    dataset="TC track matching audit",
                    source="derived", note="how each catalogue entry was matched to IBTrACS")

TRACK_NAMES = sorted(tracks_df["name"].unique())
print(f"{len(TRACK_NAMES)} storms | {len(tracks_df)} track fixes | "
      f"{tracks_df['time'].min().date()} -> {tracks_df['time'].max().date()}")
tracks_df.head()

### 4.2.1 Peak intensity per storm

Saffir-Simpson category from the peak 1-minute sustained wind along each matched track. Used to
stratify the catalogue and to pick the events worth simulating.

In [ ]:
SS_BOUNDS = [34, 64, 83, 96, 113, 137]              # 1-min sustained wind, knots
SS_LABELS = ["TD (<34 kn)", "TS (34-63)", "Cat 1 (64-82)", "Cat 2 (83-95)",
             "Cat 3 (96-112)", "Cat 4 (113-136)", "Cat 5 (>=137)"]
SS_COLORS = ["#F7A98F", "#F4795C", "#E64B35", "#C32B22", "#9C1116", "#730A12", "#47030B"]
KN_TO_KMH = 1.852

peaks = (tracks_df.groupby("name", as_index=False)["vmax_kn"].max())
cat_df = (storm_df.merge(peaks, on="name", how="inner")
                  .assign(year=lambda d: d["ref_date"].dt.year,
                          date=lambda d: d["ref_date"].dt.date,
                          cat=lambda d: np.digitize(d["vmax_kn"], SS_BOUNDS),
                          vmax_kmh=lambda d: (d["vmax_kn"] * KN_TO_KMH).round(1)))
cat_df["category"] = pd.Categorical([SS_LABELS[c] for c in cat_df["cat"]],
                                    categories=SS_LABELS, ordered=True)
cat_df["label"] = cat_df["name"] + " (" + cat_df["year"].astype(str) + ")"

no_track = sorted(set(storm_df["name"]) - set(cat_df["name"]))
if no_track:
    print(f"excluded, no matched track: {no_track}")

# per-storm detail and per-category summary
detail = (cat_df.sort_values(["cat", "year"], ascending=[False, True])
                [["name", "date", "year", "vmax_kn", "vmax_kmh", "category"]]
                .reset_index(drop=True))
summary = (cat_df.sort_values(["cat", "year"])
                 .groupby("category", observed=False)
                 .agg(n=("name", "size"),
                      vmax_kmh=("vmax_kmh", "max"),
                      storms=("label", lambda s: ", ".join(s)))
                 .reset_index())
summary["pct"] = (100 * summary["n"] / summary["n"].sum()).round(1)
summary = summary[["category", "n", "pct", "vmax_kmh", "storms"]]

summary_csv = DIRS["rainfall"] / "tc_catalogue_summary.csv"
detail_csv = DIRS["rainfall"] / "tc_catalogue_detail.csv"
summary.to_csv(summary_csv, index=False)
detail.to_csv(detail_csv, index=False)
register_output(summary_csv, dataset="TC catalogue summary", source="derived from tc_tracks_mdg.csv",
                note="storm count and peak wind per Saffir-Simpson category")
register_output(detail_csv, dataset="TC catalogue detail", source="derived from tc_tracks_mdg.csv",
                note="per-storm peak intensity and date of passage")

display(summary)
detail.head(15)

In [ ]:
if MAKE_FIGURES:
    years = np.arange(cat_df["year"].min(), cat_df["year"].max() + 1)
    counts = (cat_df.pivot_table(index="year", columns="cat", values="name", aggfunc="count")
                    .reindex(index=years, columns=range(len(SS_LABELS)))
                    .fillna(0).astype(int))

    fig, ax = plt.subplots(figsize=(14, 6))
    bottom = np.zeros(len(years))
    for c in range(len(SS_LABELS)):
        v = counts[c].values.astype(float)
        ax.bar(years, np.where(v > 0, v, np.nan), bottom=bottom, width=0.72,
               color=SS_COLORS[c], label=SS_LABELS[c], edgecolor="white", linewidth=2, zorder=3)
        bottom += v

    totals = counts.sum(axis=1).values
    for x, t in zip(years, totals):
        if t:
            ax.text(x, t + 0.12, str(t), ha="center", va="bottom", fontsize=11, color="#444444")
    ax.axhline(totals.mean(), color="#888888", lw=1.2, ls="--", zorder=2)
    ax.text(years[-1] + 0.5, totals.mean(), f"mean {totals.mean():.1f}/yr",
            va="center", ha="left", fontsize=11, color="#888888")

    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=45, ha="right")
    ax.set_yticks(np.arange(0, totals.max() + 2))
    ax.set(ylabel="Number of storms", xlabel="Year of passage over Madagascar",
           title="Tropical cyclones affecting Madagascar, by year and peak intensity")
    ax.grid(axis="y", color="#dddddd", lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], title="Peak Saffir-Simpson category",
              loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=11, frameon=False)

    plt.tight_layout()
    savefig("06_tc_catalogue_by_year")
    plt.show()

## 4.3 GPM rainfall cube and cyclone attribution

Every cell-day of the GPM record is labelled **cyclone-driven or not**, by testing it against the
matched best tracks. That single label is what lets the rainfall record be split into the three
populations the design events need — TC, non-TC, and combined — and it answers the question that
justifies building the design events around cyclone rainfall in the first place: *what share of
Madagascar's rainfall, and of its extremes, is cyclonic?*

### The attribution rule

A cell is flagged for a given day if **any** 6-hourly track fix on that day falls within
`TC_SEARCH_RADIUS_KM` (500 km) of the cell centre. So the flag is the union of 500 km disks drawn
around every fix of every storm, evaluated day by day.

500 km is generous next to a typical radius of maximum wind (~50 km), and deliberately so: it
captures the outer rainbands and the moisture convergence ahead of the centre, which is where much of
the flood-producing rainfall falls. It is the main tunable assumption in the whole rainfall chain —
widen it and the TC share rises, narrow it and TC rainfall leaks into the non-TC population.

Two consequences of working on a **daily** cube, both worth keeping in mind:

* The flag has daily granularity. A storm 400 km away at 00:00 UTC that has moved 900 km away by
  18:00 UTC still flags the whole day. Ingesting IMERG half-hourly would sharpen this.
* Where two storms are in range of the same cell on the same day, `tc_event_id` keeps only one of
  them (whichever the loop visits last, i.e. last alphabetically). This affects **0.37 %** of flagged
  cell-days in this record, and a maximum of two storms ever overlap. `tc_flag` — the variable the
  statistics actually use — is unaffected; only the *attribution to a named storm* is ambiguous in
  those cells.

### What this cell produces

**One file on disk:**

| File | Variables | Dims | Notes |
|---|---|---|---|
| `processed/rainfall/gpm_daily_tc_attribution.nc` | `tc_flag` (`int8`, 0/1)<br>`tc_event_id` (`int16`, 0 = none) | `(time, lat, lon)` = 9 669 × 149 × 84 | zlib-compressed, ~1 MB. `tc_event_id` indexes the `storm_names` attribute, 1-based. |

Storm identity is stored as an **integer index rather than a string**: a fixed-width character array
over `(time, lat, lon)` inflates the same file to several GB.

**In-memory objects the rest of section 4 depends on** (not written to disk here):

| Object | Meaning |
|---|---|
| `tc_flag` | boolean mask, `(time, lat, lon)` |
| `precip_tc`, `precip_nontc` | the daily cube masked to each population — **the inputs to the per-pixel frequency analysis** |
| `total_precip`, `tc_precip_total` | record-total accumulation maps (mm) |
| `tc_fraction_map`, `tc_fraction` | cyclone share of rainfall, per pixel and nationally |
| `n_years` | record length, for converting totals to annual means |

The cell is cached: once the netCDF exists it is read back instead of recomputed (~2 min), so
re-running the notebook is cheap. Delete the file or set `OVERWRITE = True` to rebuild.

In [ ]:
ds = xr.open_dataset(GPM_FILE)
ds = ds.rename({"latitude": "lat", "longitude": "lon"}).transpose("time", "lat", "lon")

lat_grid = ds["lat"].values
lon_grid = ds["lon"].values
time_grid = pd.to_datetime(ds["time"].values)
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

print(f"GPM cube : {ds['precip'].shape} (time, lat, lon)")
print(f"  period : {time_grid.min().date()} -> {time_grid.max().date()} ({len(time_grid)} days)")
print(f"  grid   : {lat_grid.min():.2f} to {lat_grid.max():.2f} lat, "
      f"{lon_grid.min():.2f} to {lon_grid.max():.2f} lon @ "
      f"{abs(lat_grid[1] - lat_grid[0]):.2f} deg")
print(f"  units  : mm/day (variable 'precip')")


def haversine_km(lat1, lon1, lat2, lon2, radius_km: float = 6371.0):
    """Great-circle distance on a sphere, broadcast over arrays."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))

In [ ]:
# =============================================================================
# Cyclone attribution of the daily rainfall record
#   in  : tracks_df (section 4.2), ds / lat2d / lon2d / time_grid (section 4.3)
#   out : processed/rainfall/gpm_daily_tc_attribution.nc
#         + the population splits used by every later rainfall cell
# =============================================================================
tc_flag_nc = DIRS["rainfall"] / "gpm_daily_tc_attribution.nc"

# 1. Storm name -> integer code. Stored as an index, never as text: a fixed-width
#    character array over (time, lat, lon) would inflate the netCDF to several GB,
#    while int16 compresses to ~1 MB because the field is overwhelmingly zero.
STORM_INDEX = {name: i + 1 for i, name in enumerate(TRACK_NAMES)}     # 0 = no cyclone

if tc_flag_nc.exists() and not OVERWRITE:
    # 2a. Cached path: the mask takes ~2 min to build, so it is read back verbatim.
    tc_ds = xr.open_dataset(tc_flag_nc)
    print(f"loaded cached TC attribution: {tc_flag_nc.relative_to(EXEC_DIR)}")
else:
    # 2b. Build path. tc_id holds the storm code per cell-day; 0 means "no cyclone
    #     within TC_SEARCH_RADIUS_KM on that day".
    t0 = time.time()
    tc_id = np.zeros((len(time_grid), len(lat_grid), len(lon_grid)), dtype="int16")
    time_index = pd.Index(time_grid)          # O(1) date -> time-axis position

    # 3. Stamp the mask: for every storm, for every day it was active, for every
    #    6-hourly fix on that day, flag the cells within the search radius.
    #    Three nested loops but each innermost step is one vectorised haversine over
    #    the whole 149 x 84 grid, so the cost is ~5k grid evaluations, not per-cell work.
    for name, grp in tracks_df.groupby("name"):
        storm_code = STORM_INDEX[name]
        for day, fixes in grp.groupby(grp["time"].dt.floor("D")):
            if pd.Timestamp(day) not in time_index:
                continue                      # track day outside the GPM record
            t_idx = time_index.get_loc(pd.Timestamp(day))
            for lat_fix, lon_fix in zip(fixes["lat"].values, fixes["lon"].values):
                within = haversine_km(lat2d, lon2d, lat_fix, lon_fix) <= TC_SEARCH_RADIUS_KM
                # union across fixes and storms; on the rare cell-day reached by two
                # storms (0.4% of flagged cells) the later storm wins the *label*,
                # but the cell stays flagged either way
                tc_id[t_idx][within] = storm_code

    # 4. Wrap as a dataset on the GPM coordinates and write it compressed.
    #    tc_flag is the boolean the statistics use; tc_event_id keeps the attribution
    #    so any flagged cell-day can be traced back to a named storm.
    tc_ds = xr.Dataset(
        {"tc_flag": (("time", "lat", "lon"), (tc_id > 0).astype("int8")),
         "tc_event_id": (("time", "lat", "lon"), tc_id)},
        coords={"time": ds["time"], "lat": ds["lat"], "lon": ds["lon"]},
        attrs={"description": "cell-days within TC_SEARCH_RADIUS_KM of a matched IBTrACS fix",
               "radius_km": TC_SEARCH_RADIUS_KM,
               "storm_names": ", ".join(TRACK_NAMES),
               "storm_name_lookup": "tc_event_id = index into storm_names, 1-based; 0 = none",
               "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")},
    )
    tc_ds.to_netcdf(tc_flag_nc, encoding={v: {"zlib": True, "complevel": 4}
                                          for v in ("tc_flag", "tc_event_id")})
    print(f"built in {time.time() - t0:.0f}s -> {human(tc_flag_nc.stat().st_size)}")
    register_output(tc_flag_nc, dataset="TC rainfall attribution mask",
                    source="derived from GPM daily + IBTrACS tracks",
                    note=f"{TC_SEARCH_RADIUS_KM} km moving radius around each 6-hourly fix")

# 5. Split the record into the three rainfall populations. These are lazy xarray
#    views, not copies - nothing is computed until a statistic is requested.
#    They are the inputs to the per-pixel frequency analysis in section 4.6.
tc_flag = tc_ds["tc_flag"].astype(bool)
precip_tc = ds["precip"].where(tc_flag)              # cyclone-driven rainfall
precip_nontc = ds["precip"].where(~tc_flag)          # everything else
                                                     # (combined = ds["precip"] itself)

# 6. Record-length diagnostics: how much rain fell, and how much of it was cyclonic.
total_precip = ds["precip"].sum("time", skipna=True)          # mm over the record
tc_precip_total = precip_tc.sum("time", skipna=True)          # mm, cyclonic only
tc_fraction_map = (tc_precip_total / total_precip.where(total_precip > 0)).rename("tc_fraction")
tc_fraction = float(tc_precip_total.sum() / total_precip.sum())
n_years = (time_grid.max() - time_grid.min()).days / 365.25

print(f"TC-attributed fraction of total rainfall : {tc_fraction:.1%}")
print(f"days with at least one TC-flagged cell   : "
      f"{int(tc_flag.any(dim=('lat', 'lon')).sum())} "
      f"over {n_years:.1f} years")
print(f"mean annual rainfall (AOI mean)          : "
      f"{float(total_precip.mean()) / n_years:.0f} mm/yr")
print(f"  of which cyclonic                      : "
      f"{float(tc_precip_total.mean()) / n_years:.0f} mm/yr")
print(f"\nwritten   : {tc_flag_nc.relative_to(EXEC_DIR)}")
print(f"in memory : tc_flag, precip_tc, precip_nontc, total_precip, tc_precip_total, "
      f"tc_fraction_map, tc_fraction, n_years")

In [ ]:
if MAKE_FIGURES:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    (total_precip / n_years).plot(ax=axes[0], cmap="YlGnBu", vmin=0,
                                  cbar_kwargs={"label": "mm/yr"})
    axes[0].set_title(f"Mean annual rainfall\n({time_grid.min().year}-{time_grid.max().year})")

    (tc_precip_total / n_years).plot(ax=axes[1], cmap="YlOrRd", vmin=0,
                                     cbar_kwargs={"label": "mm/yr"})
    axes[1].set_title("Cyclone-attributed rainfall")

    (100 * tc_fraction_map).plot(ax=axes[2], cmap="YlOrRd", vmin=0, vmax=40,
                                 cbar_kwargs={"label": "%"})
    axes[2].set_title(f"Cyclone-attributed share\n(national mean {tc_fraction:.1%})")

    for ax in axes:
        gdf_catch.boundary.plot(ax=ax, color="0.3", linewidth=0.3, alpha=0.7)
        ax.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)")
    plt.tight_layout()
    savefig("07_rainfall_tc_attribution")
    plt.show()

### 4.3.1 Persisted population records and the storm lookup

The rainfall populations are kept on disk as the auditable inputs to the extreme-value analysis, so a
fit can be reproduced or reviewed without re-deriving the split:

| File | Content | Size |
|---|---|---|
| `precip_tc.nc` | daily rainfall on cyclone-flagged cell-days | ~5 MB |
| `precip_nontc.nc` | daily rainfall on all other cell-days | ~176 MB |
| *combined* | **not written** — it is the unmodified input, `Data/gpm_daily_precipitation_pp.nc` | 183 MB |

**`NaN` does not mean zero.** In `precip_tc.nc`, `NaN` marks a cell-day belonging to the *other*
population; a cyclone-flagged day with no rain is a genuine `0.0`. Every downstream statistic must
therefore be NaN-aware, and a count of non-NaN values gives the sample size available at that pixel —
which is exactly what the zero-inflation term of the frequency analysis needs. The two files are
complementary and lossless: at every cell-day exactly one of them is non-NaN, and together they
reproduce the input cube.

The size asymmetry is not a mistake — only **1.22 %** of cell-days are cyclone-flagged, so
`precip_tc.nc` is almost entirely NaN and compresses 484 MB down to ~5 MB, while `precip_nontc.nc`
carries nearly the whole record.

#### The storm lookup

`tc_event_lookup.csv` is the key that ties the integer codes in `tc_event_id` back to named storms,
and the place to read event windows from when setting up model runs. One row per matched storm:

| Column | Meaning |
|---|---|
| `tc_event_id` | integer code stored in `gpm_daily_tc_attribution.nc` |
| `name`, `sid`, `season` | catalogue name, IBTrACS storm id, cyclone season |
| `ref_date` | catalogue date of passage over Madagascar |
| `track_start`, `track_end` | first and last best-track fix |
| `event_start`, `event_end` | **the forcing window** — track extent padded by `EVENT_PAD_DAYS` |
| `vmax_kn`, `vmax_kmh`, `category` | peak intensity and Saffir-Simpson class |
| `n_fixes` | best-track fixes used |
| `flagged_days`, `first_flagged_day`, `last_flagged_day` | days on which the storm actually appears in the attribution cube |
| `flagged_cell_days` | cells x days attributed to this storm — a proxy for its rainfall footprint |

`flagged_days` can be shorter than the track: a storm only enters the cube on days when it came
within `TC_SEARCH_RADIUS_KM` of the AOI grid. A storm with `flagged_days = 0` was matched in IBTrACS
but never came close enough to attribute any rainfall — worth checking before it is used as a
historical event.

In [ ]:
# =============================================================================
# Persist the rainfall populations and the storm lookup
#   in  : precip_tc / precip_nontc (section 4.3), tc_ds, tracks_df, cat_df
#   out : processed/rainfall/precip_tc.nc
#         processed/rainfall/precip_nontc.nc
#         processed/rainfall/tc_event_lookup.csv
# =============================================================================

def event_window_dates(times, pad_days: int = EVENT_PAD_DAYS):
    """Forcing window for an event: whole days spanning the track, padded either side.

    Defined once here and reused by ``event_window`` in section 4.4, so the windows
    written to the lookup table cannot drift from the windows the forcing files use.
    """
    return (times.min().floor("D") - pd.Timedelta(days=pad_days),
            times.max().floor("D") + pd.Timedelta(days=pad_days))


# --- 1. the two population records ------------------------------------------
#     NaN means "belongs to the other population", NOT zero rainfall: a dry day
#     inside the population is a genuine 0.0. Downstream stats must be NaN-aware.
population_records = {
    "tc": (DIRS["rainfall"] / "precip_tc.nc", precip_tc,
           f"cyclone-driven daily rainfall: cell-days within {TC_SEARCH_RADIUS_KM} km "
           f"of a matched IBTrACS fix"),
    "nontc": (DIRS["rainfall"] / "precip_nontc.nc", precip_nontc,
              f"non-cyclonic daily rainfall: cell-days NOT within {TC_SEARCH_RADIUS_KM} km "
              f"of a matched IBTrACS fix"),
}

for pop, (path, series, description) in population_records.items():
    if path.exists() and not OVERWRITE:
        print(f"  cached   {path.name}  ({human(path.stat().st_size)})")
        continue
    t0 = time.time()
    out = series.astype("float32").rename("precip").to_dataset()
    out["precip"].attrs = {
        "units": "mm/day",
        "long_name": f"{pop} daily precipitation",
        "description": description,
        "nan_meaning": "cell-day belongs to the complementary population - NOT zero rainfall",
    }
    out.attrs = {
        "title": f"GPM IMERG daily rainfall, {pop} population",
        "population": pop,
        "tc_search_radius_km": TC_SEARCH_RADIUS_KM,
        "source_cube": GPM_FILE.name,
        "attribution_mask": tc_flag_nc.name,
        "complement": "precip_nontc.nc" if pop == "tc" else "precip_tc.nc",
        "combined_population": f"the unmodified input cube, {GPM_FILE.name}",
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    out.to_netcdf(path, encoding={"precip": {"zlib": True, "complevel": 4}})
    print(f"  written  {path.name}  ({human(path.stat().st_size)} in {time.time() - t0:.0f}s)")
    register_output(path, dataset=f"rainfall population ({pop})",
                    source=f"{GPM_FILE.name} masked by {tc_flag_nc.name}",
                    note=description + "; NaN = other population, not zero rainfall")

# --- 2. per-storm footprint in the attribution cube --------------------------
#     One pass over the time axis, counting cells per storm code. Cheaper than
#     evaluating (tc_event_id == code) once per storm over the whole cube.
codes = tc_ds["tc_event_id"].values
n_codes = len(TRACK_NAMES) + 1                        # +1 because code 0 = "no cyclone"
cell_days = np.zeros(n_codes, dtype="int64")
day_present = np.zeros((n_codes, len(time_grid)), dtype=bool)
for t_idx in range(codes.shape[0]):
    present, counts = np.unique(codes[t_idx], return_counts=True)
    cell_days[present] += counts
    day_present[present, t_idx] = True

# --- 3. assemble the lookup --------------------------------------------------
rows = []
for name in TRACK_NAMES:
    code = STORM_INDEX[name]
    track = tracks_df[tracks_df["name"] == name].sort_values("time")
    win_start, win_end = event_window_dates(track["time"])
    flagged = time_grid[day_present[code]]
    rows.append({
        "tc_event_id": code,
        "name": name,
        "sid": track["sid"].iloc[0] if "sid" in track.columns else "",
        "season": int(track["season"].iloc[0]) if "season" in track.columns else None,
        "track_start": track["time"].min(),
        "track_end": track["time"].max(),
        "event_start": win_start,
        "event_end": win_end,
        "n_fixes": len(track),
        "vmax_kn": round(float(track["vmax_kn"].max()), 1),
        "vmax_kmh": round(float(track["vmax_kn"].max()) * KN_TO_KMH, 1),
        "flagged_days": int(day_present[code].sum()),
        "first_flagged_day": flagged.min() if len(flagged) else pd.NaT,
        "last_flagged_day": flagged.max() if len(flagged) else pd.NaT,
        "flagged_cell_days": int(cell_days[code]),
    })

tc_lookup = (pd.DataFrame(rows)
             .merge(storm_df, on="name", how="left")                    # ref_date
             .merge(cat_df[["name", "category"]], on="name", how="left"))
tc_lookup = tc_lookup[["tc_event_id", "name", "sid", "season", "ref_date",
                       "track_start", "track_end", "event_start", "event_end",
                       "n_fixes", "vmax_kn", "vmax_kmh", "category",
                       "flagged_days", "first_flagged_day", "last_flagged_day",
                       "flagged_cell_days"]].sort_values("tc_event_id")

lookup_csv = DIRS["rainfall"] / "tc_event_lookup.csv"
tc_lookup.to_csv(lookup_csv, index=False)
register_output(lookup_csv, dataset="TC event lookup",
                source="derived from tc_tracks_mdg.csv + gpm_daily_tc_attribution.nc",
                note="storm name <-> tc_event_id <-> track and forcing window, plus the "
                     "footprint each storm has in the attribution cube")

never_flagged = tc_lookup.loc[tc_lookup["flagged_days"] == 0, "name"].tolist()
print(f"  written  {lookup_csv.name}  ({len(tc_lookup)} storms)")
print(f"unattributed cell-days (code 0)  : {cell_days[0]:,} "
      f"({100 * cell_days[0] / cell_days.sum():.1f}% of the cube)")
print(f"storms never within {TC_SEARCH_RADIUS_KM} km of the grid : "
      f"{never_flagged if never_flagged else 'none'}")
print(f"widest footprint                 : "
      f"{tc_lookup.loc[tc_lookup['flagged_cell_days'].idxmax(), 'name']} "
      f"({tc_lookup['flagged_cell_days'].max():,} cell-days)")
tc_lookup.head(10)

## 4.4 Event rainfall and swath — worked example

For one storm: aggregate the GPM record over the event window (best track extent ± `EVENT_PAD_DAYS`),
build the `SWATH_RADIUS_KM` swath as the union of geodesic disks along the hourly-interpolated track,
and report the accumulation statistics inside it.

The interpolation to hourly steps matters: at the native 6-hourly fix spacing the union of disks is
visibly scalloped and under-covers the area between fixes.

`event_swath` measures elapsed time in seconds rather than taking an integer view of the timestamps.
That is deliberate — `datetime64` integer units are not stable across pandas versions (3.x parses CSV
timestamps as microseconds, 2.x gave nanoseconds), and a hard-coded unit does not error, it silently
shrinks the track to a single fix. The function now raises instead.

Change `STORM_NAME` to inspect any storm in `TRACK_NAMES`.

In [ ]:
from shapely.geometry import Polygon
from shapely.ops import unary_union
from pyproj import Geod

# same sphere as haversine_km, so the polygon and the raster mask agree exactly
_GEOD = Geod(a=6371000.0, f=0.0)


def geodesic_circle(lat: float, lon: float, radius_km: float, n: int = 120) -> Polygon:
    az = np.linspace(0, 360, n, endpoint=False)
    lons, lats, _ = _GEOD.fwd(np.full(n, lon), np.full(n, lat), az,
                              np.full(n, radius_km * 1000.0))
    return Polygon(np.column_stack([lons, lats]))


def event_window(name: str, pad_days: int = EVENT_PAD_DAYS):
    """(day_start, day_end, track subset) for a storm, from its best-track extent.

    Delegates the padding to ``event_window_dates`` (section 4.3.1), so the windows
    recorded in tc_event_lookup.csv are by construction the windows the forcing files
    in section 4.5 are cut to.
    """
    tr = tracks_df[tracks_df["name"] == name].sort_values("time")
    if tr.empty:
        raise KeyError(f"{name} not in the matched catalogue: {TRACK_NAMES}")
    day_start, day_end = event_window_dates(tr["time"], pad_days)
    return day_start, day_end, tr


def event_swath(tr: pd.DataFrame, radius_km: float = SWATH_RADIUS_KM,
                densify_min: int = 60):
    """(boolean mask on the GPM grid, swath polygon) from a densified track.

    The track is interpolated to *densify_min* steps before the disks are unioned: at the
    native 6-hourly fix spacing the swath edge is visibly scalloped and under-covers the
    area between fixes.

    Elapsed time is computed in seconds via ``np.timedelta64``, never by taking an integer
    view of the datetimes. The integer unit of ``datetime64`` is not stable - pandas 3
    parses CSV timestamps as microseconds where pandas 2 gave nanoseconds - and a
    hard-coded unit silently collapses the densified track to a single point instead of
    raising.
    """
    times = tr["time"].to_numpy()
    seconds = (times - times[0]) / np.timedelta64(1, "s")            # float, unit-safe
    step = densify_min * 60.0
    dense = np.arange(0.0, seconds[-1] + step, step)
    if dense.size < max(2, seconds[-1] / step):                      # guard the unit bug
        raise ValueError(f"track densification produced only {dense.size} points for a "
                         f"{seconds[-1] / 3600:.1f} h track - check the time units")
    lon_dense = np.interp(dense, seconds, tr["lon"].values)
    lat_dense = np.interp(dense, seconds, tr["lat"].values)

    dist_min = np.full(lat2d.shape, np.inf)
    for la, lo in zip(lat_dense, lon_dense):
        np.minimum(dist_min, haversine_km(lat2d, lon2d, la, lo), out=dist_min)

    poly = unary_union([geodesic_circle(la, lo, radius_km)
                        for la, lo in zip(lat_dense, lon_dense)])
    return dist_min <= radius_km, gpd.GeoDataFrame(geometry=[poly], crs=WORK_CRS)


STORM_NAME = "ENAWO"
MASK_TO_SWATH = False            # True -> show accumulation only inside the swath

day_start, day_end, track = event_window(STORM_NAME)
precip_evt = ds["precip"].sel(time=slice(day_start, day_end + pd.Timedelta(hours=23, minutes=59)))
if precip_evt.sizes["time"] == 0:
    raise ValueError(f"no GPM steps between {day_start.date()} and {day_end.date()} "
                     f"(cube spans {time_grid.min().date()} to {time_grid.max().date()})")
precip_cum = precip_evt.sum("time", skipna=True)
swath_mask, gdf_swath = event_swath(track)

cum = precip_cum.values
in_swath = cum[swath_mask]
iy, ix = np.unravel_index(np.nanargmax(np.where(swath_mask, cum, np.nan)), cum.shape)
print(f"{STORM_NAME}: track {track['time'].min()} -> {track['time'].max()}")
print(f"  event window   {day_start.date()} -> {day_end.date()} "
      f"({precip_evt.sizes['time']} days)")
print(f"  peak intensity {track['vmax_kn'].max():.0f} kn "
      f"({track['vmax_kn'].max() * KN_TO_KMH:.0f} km/h)")
print(f"  swath          {SWATH_RADIUS_KM} km radius, {swath_mask.sum()} GPM cells")
print(f"  accumulation   mean {np.nanmean(in_swath):.1f} mm | "
      f"max {np.nanmax(in_swath):.1f} mm at ({lat_grid[iy]:.2f}, {lon_grid[ix]:.2f})")
(precip_evt.where(xr.DataArray(swath_mask, dims=("lat", "lon"),
                               coords={"lat": lat_grid, "lon": lon_grid}))
           .mean(("lat", "lon")).to_series().round(1).rename("swath-mean mm/day"))

In [ ]:
if MAKE_FIGURES:
    field = np.where(swath_mask, cum, np.nan) if MASK_TO_SWATH else cum
    levels = np.array([1, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600])
    cmap = plt.get_cmap("YlGnBu", len(levels))
    norm = mcolors.BoundaryNorm(levels, cmap.N, extend="max")

    fig, ax = plt.subplots(figsize=(11, 10))
    im = ax.pcolormesh(lon2d, lat2d, np.ma.masked_less(np.nan_to_num(field), levels[0]),
                       cmap=cmap, norm=norm, shading="auto", alpha=0.85, zorder=2)
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, extend="max", ticks=levels)
    cbar.set_label("Event accumulated precipitation (mm)")

    gdf_swath.boundary.plot(ax=ax, color="red", linewidth=3, zorder=4)
    gdf_catch.dissolve().boundary.plot(ax=ax, color="white", linewidth=1.8, zorder=3)

    ax.plot(track["lon"], track["lat"], color="black", lw=2.5, ls="--", zorder=5)
    ax.scatter(track["lon"], track["lat"], color="black", s=20, zorder=5)
    ax.scatter(track["lon"].iloc[0], track["lat"].iloc[0], marker="o", s=140,
               facecolor="white", edgecolor="black", lw=1.5, zorder=6)     # genesis
    ax.scatter(track["lon"].iloc[-1], track["lat"].iloc[-1], marker="X", s=180,
               facecolor="white", edgecolor="black", lw=1.5, zorder=6)     # dissipation

    add_basemap(ax)
    pad = 3
    ax.set_xlim(lon_grid.min() - pad, lon_grid.max() + pad)
    ax.set_ylim(lat_grid.min() - pad, lat_grid.max() + pad)
    ax.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)")
    ax.set_title(f"{STORM_NAME} - event accumulation and {SWATH_RADIUS_KM} km swath\n"
                 f"{day_start.date()} to {day_end.date()} "
                 f"({precip_evt.sizes['time']} days)", fontsize=16)
    ax.legend(handles=[
        Line2D([], [], color="red", lw=3, label=f"{SWATH_RADIUS_KM} km swath"),
        Line2D([], [], color="black", lw=2.5, ls="--", marker="o", ms=5, label="Best track"),
        Line2D([], [], ls="", marker="o", markerfacecolor="white", markeredgecolor="black",
               label="Genesis"),
        Line2D([], [], ls="", marker="X", markerfacecolor="white", markeredgecolor="black",
               label="Dissipation"),
    ], loc="lower left", fontsize=11, framealpha=0.9)

    label = ax.text(track["lon"].mean(), track["lat"].mean() + 1, STORM_NAME,
                    fontsize=18, color="white", ha="center", va="center", zorder=7)
    label.set_path_effects([path_effects.Stroke(linewidth=4, foreground="black"),
                            path_effects.Normal()])
    plt.tight_layout()
    savefig(f"08_event_swath_{STORM_NAME.lower()}")
    plt.show()

## 4.6 Design hyetographs

For the hazard maps the model is driven by synthetic events of a given return period rather than by
one historical storm. Three steps, all on the GPM grid:

1. **Frequency analysis.** Annual maxima of daily rainfall per cell, taken over the **cyclone season
   year (1 July → 30 June)** so a single season is never split across two samples, fitted with a
   **Gumbel (EV1) distribution by the method of moments**. Gives the 24-hour depth for each return
   period.
2. **Depth-duration disaggregation.** With no sub-daily observations in hand, the shorter-duration
   depths are obtained from the empirical power law
   *P(t) = P(24 h) · (t / 24)<sup>n</sup>*, `n = 1/3` (the IMD relation, standard practice for
   monsoon and tropical-cyclone climates where only daily gauges exist).
3. **Alternating Block Method.** The incremental depths of successive durations are ordered with the
   peak in the middle and alternating blocks either side — the standard design hyetograph shape
   (Chow, Maidment & Mays, *Applied Hydrology*, §14.4).

> **Assumptions to revisit — this section is a scaffold, not a finished design event.**
>
> * The power-law exponent *n* is the weakest link: it sets the peak intensity, which sets the
>   modelled flood depth. At `n = 1/3` the implied 1 h / 24 h depth ratio is 0.35, which is
>   reasonable for the tropics, but the extrapolation below 1 h is unconstrained and can produce
>   implausible 30-minute blocks. The cell prints a warning when the peak block exceeds
>   `PLAUSIBLE_PEAK_MM_HR`. Calibrate *n* against IMERG half-hourly, or against any sub-daily gauge
>   record for Madagascar, before the design results are used for anything but ranking.
> * The Gumbel fit assumes stationarity over 1998–2024. Twenty-six seasons is a short record for a
>   100-year estimate, and single-pixel maxima in the tail are unreliable — hence the spatial
>   quantile below rather than the field maximum. Treat the 100-year depths as indicative.
> * The alternating block shape assumes one centred peak. A slow-moving cyclone with several
>   rainbands has a multi-peaked hyetograph that this cannot represent; for those, drive the model
>   with the observed event (section 4.5).

In [ ]:
RETURN_PERIODS = (2, 5, 10, 25, 50, 100)
HYETOGRAPH_RP = 100                # return period used for the exported hyetograph
HYETOGRAPH_QUANTILE = 0.50         # spatial statistic of the depth field: 0.5 = AOI median
HYETOGRAPH_STEP_MIN = 30           # target SFINCS forcing step
HYETOGRAPH_DURATION_H = 24
DD_EXPONENT = 1 / 3                # P(t) = P(24h) * (t/24)^n  -> P(1h)/P(24h) = 0.35
SEASON_START_MONTH = 7             # cyclone-season year starts 1 July
PLAUSIBLE_PEAK_MM_HR = 150         # sanity threshold on the peak block intensity

# --- 1. Gumbel fit on seasonal maxima of daily rainfall --------------------
season = ds["time"].dt.year + (ds["time"].dt.month >= SEASON_START_MONTH)
annual_max = ds["precip"].groupby(season.rename("season")).max("time")
complete = annual_max["season"][1:-1]                    # drop the two partial seasons
annual_max = annual_max.sel(season=complete)

mu_s = annual_max.mean("season")
sd_s = annual_max.std("season", ddof=1)
beta = sd_s * np.sqrt(6) / np.pi                         # Gumbel scale
mu = mu_s - 0.5772 * beta                                # Gumbel location

rp = xr.DataArray(list(RETURN_PERIODS), dims="return_period",
                  coords={"return_period": list(RETURN_PERIODS)})
depth_24h = (mu - beta * np.log(-np.log(1 - 1 / rp))).transpose("return_period", "lat", "lon")
depth_24h.name = "depth_24h"
depth_24h.attrs = {"units": "mm", "long_name": "24-hour rainfall depth",
                   "method": "Gumbel (EV1), method of moments, seasonal maxima",
                   "seasons": int(annual_max.sizes["season"]),
                   "season_start_month": SEASON_START_MONTH}

idf_nc = DIRS["rainfall"] / "design_depth_24h_gumbel.nc"
if OVERWRITE or not idf_nc.exists():
    depth_24h.to_dataset().to_netcdf(idf_nc)
register_output(idf_nc, dataset="design 24 h rainfall depth",
                source="GPM IMERG daily, Gumbel fit on seasonal maxima",
                note=f"return periods {RETURN_PERIODS}, {int(annual_max.sizes['season'])} seasons")

print(f"seasonal maxima : {int(annual_max.sizes['season'])} complete seasons "
      f"({int(complete.min())}-{int(complete.max())})")
print("24 h design depth over the AOI (mm):")
print(pd.DataFrame({
    "return_period": list(RETURN_PERIODS),
    "AOI_mean": depth_24h.mean(("lat", "lon")).round(0).values,
    "AOI_p95": depth_24h.quantile(0.95, ("lat", "lon")).round(0).values,
    "AOI_max": depth_24h.max(("lat", "lon")).round(0).values,
}).to_string(index=False))

In [ ]:
def design_hyetograph(depth_24h_mm: float, *, duration_h: int = HYETOGRAPH_DURATION_H,
                      step_min: int = HYETOGRAPH_STEP_MIN,
                      exponent: float = DD_EXPONENT) -> pd.DataFrame:
    """Alternating Block Method hyetograph from a 24-hour depth.

    Depth-duration curve: P(t) = P(24h) * (t/24)**exponent.
    Returns a frame with the time-ordered blocks in mm and mm/hr.
    """
    step_h = step_min / 60
    n_blocks = int(round(duration_h / step_h))
    durations = np.arange(1, n_blocks + 1) * step_h
    cumulative = depth_24h_mm * (durations / 24.0) ** exponent
    increments = np.diff(cumulative, prepend=0.0)          # descending by construction

    # alternating blocks: largest in the middle, then right, left, right, ...
    order = np.empty(n_blocks, dtype=int)
    centre = n_blocks // 2
    right, left = centre, centre - 1
    for i in range(n_blocks):
        if i % 2 == 0:
            order[right] = i
            right += 1
        else:
            order[left] = i
            left -= 1
    blocks = increments[order]

    return pd.DataFrame({
        "time_h": np.arange(n_blocks) * step_h,
        "depth_mm": blocks,
        "intensity_mm_hr": blocks / step_h,
    })


# One nationally representative hyetograph is exported as a template. For the production
# runs, feed each catchment its own depth: depth_24h is a raster, so
#   d = float(depth_24h.sel(return_period=RP).rio.clip(catchment_geom).median())
# and call design_hyetograph(d) per domain.
rp_field = depth_24h.sel(return_period=HYETOGRAPH_RP)
rp_depth = float(rp_field.quantile(HYETOGRAPH_QUANTILE, ("lat", "lon")))
hyeto = design_hyetograph(rp_depth)

hyeto_csv = DIRS["rainfall"] / f"design_hyetograph_rp{HYETOGRAPH_RP}_{HYETOGRAPH_STEP_MIN}min.csv"
hyeto.round(3).to_csv(hyeto_csv, index=False)
register_output(hyeto_csv, dataset="design hyetograph",
                source="Gumbel 24 h depth + IMD depth-duration law + alternating block method",
                note=f"RP {HYETOGRAPH_RP} yr, AOI q{HYETOGRAPH_QUANTILE:.2f} depth "
                     f"{rp_depth:.0f} mm, {HYETOGRAPH_STEP_MIN} min blocks, "
                     f"exponent n={DD_EXPONENT:.3f}")

peak = hyeto["intensity_mm_hr"].max()
print(f"RP {HYETOGRAPH_RP} yr, AOI q{HYETOGRAPH_QUANTILE:.2f} 24 h depth : {rp_depth:.0f} mm "
      f"(AOI range {float(rp_field.min()):.0f} - {float(rp_field.max()):.0f} mm)")
print(f"hyetograph  : {len(hyeto)} blocks of {HYETOGRAPH_STEP_MIN} min "
      f"({HYETOGRAPH_DURATION_H} h total)")
print(f"  total     : {hyeto['depth_mm'].sum():.1f} mm (closes on the 24 h depth)")
print(f"  1 h depth : {hyeto['depth_mm'].nlargest(2).sum():.1f} mm "
      f"({100 * hyeto['depth_mm'].nlargest(2).sum() / rp_depth:.0f}% of the 24 h depth)")
print(f"  peak      : {peak:.1f} mm/hr at t = "
      f"{hyeto.loc[hyeto['intensity_mm_hr'].idxmax(), 'time_h']:.1f} h")
if peak > PLAUSIBLE_PEAK_MM_HR:
    print(f"  !! peak block exceeds {PLAUSIBLE_PEAK_MM_HR} mm/hr - the depth-duration "
          f"extrapolation below 1 h is the likely cause.")
    print(f"     Raise DD_EXPONENT or lower HYETOGRAPH_QUANTILE, and calibrate against "
          f"IMERG half-hourly before using this.")
hyeto.head()

In [ ]:
if MAKE_FIGURES:
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5),
                             gridspec_kw={"width_ratios": [1, 1, 1.15]})

    # design depth map
    depth_24h.sel(return_period=HYETOGRAPH_RP).plot(
        ax=axes[0], cmap="YlGnBu", cbar_kwargs={"label": "mm"})
    gdf_catch.boundary.plot(ax=axes[0], color="0.3", lw=0.3, alpha=0.7)
    axes[0].set(xlabel="Longitude (deg)", ylabel="Latitude (deg)",
                title=f"{HYETOGRAPH_RP}-year 24 h rainfall depth")

    # growth curve
    for q, lbl, ls in [(0.5, "AOI median", "-"), (0.95, "AOI p95", "--"), (1.0, "AOI max", ":")]:
        vals = [float(depth_24h.sel(return_period=t).quantile(q, ("lat", "lon")))
                for t in RETURN_PERIODS]
        axes[1].plot(RETURN_PERIODS, vals, ls, marker="o", label=lbl)
    axes[1].set(xscale="log", xlabel="Return period (years)", ylabel="24 h depth (mm)",
                title="Gumbel growth curve")
    axes[1].set_xticks(RETURN_PERIODS)
    axes[1].set_xticklabels(RETURN_PERIODS)
    axes[1].legend(frameon=False)
    axes[1].grid(which="both", color="0.92")
    axes[1].set_axisbelow(True)

    # hyetograph
    axes[2].bar(hyeto["time_h"], hyeto["intensity_mm_hr"],
                width=HYETOGRAPH_STEP_MIN / 60 * 0.9, color="#1f6f8b", edgecolor="white")
    ax2 = axes[2].twinx()
    ax2.plot(hyeto["time_h"], hyeto["depth_mm"].cumsum(), color="#d95f0e", lw=2)
    ax2.set_ylabel("Cumulative depth (mm)", color="#d95f0e")
    ax2.tick_params(axis="y", colors="#d95f0e")
    axes[2].set(xlabel="Time (h)", ylabel="Intensity (mm/hr)",
                title=f"Design hyetograph, RP {HYETOGRAPH_RP} yr\n"
                      f"{rp_depth:.0f} mm in {HYETOGRAPH_DURATION_H} h, "
                      f"{HYETOGRAPH_STEP_MIN} min blocks")
    for side in ("top",):
        axes[2].spines[side].set_visible(False)

    plt.tight_layout()
    savefig("09_design_rainfall")
    plt.show()

---

# 5. HydroMT-SFINCS data catalogue and provenance

## 5.1 Data catalogue

`sfincs_data_catalog.yml` is the handover artefact: it names every grid produced above in the form
HydroMT expects, so the model build reads

```python
from hydromt_sfincs import SfincsModel

sf = SfincsModel(root="models/<catchment_id>", mode="w",
                 data_libs=["Data/processed/sfincs_data_catalog.yml"])
sf.setup_grid_from_region(region={"geom": "pluvial_catchments_mdg"}, res=50, crs="utm")
sf.setup_dep(datasets_dep=[{"elevtn": "fabdem"}])
sf.setup_subgrid(datasets_dep=[{"elevtn": "fabdem"}],
                 datasets_rgh=[{"lulc": "vito_lulc", "reclass_table": "vito_reclass"}],
                 nr_subgrid_pixels=5)
sf.setup_cn_infiltration(cn="gcn250_arcii", antecedent_moisture="avg")
sf.setup_precip_forcing_from_grid(precip="precip_event_enawo", aggregate=False)
```

Paths in the catalogue are **relative to the catalogue file**, so `Data/processed/` can be copied or
moved as a unit.

In [ ]:
catalog_path = PROC_DIR / "sfincs_data_catalog.yml"


def _rel(path: Path) -> str:
    return str(Path(path).resolve().relative_to(PROC_DIR.resolve())).replace(os.sep, "/")


meta_common = {"category": "topography", "paper_ref": "", "source_license": "", "source_url": ""}

entries = {
    "fabdem": {
        "path": _rel(dem_vrt), "data_type": "RasterDataset", "driver": "raster",
        "crs": 4326, "nodata": FABDEM_NODATA,
        "rename": {"fabdem_mdg": "elevtn"},
        "meta": {"category": "topography", "unit": "m+EGM2008", "version": "V1-2",
                 "source_url": f"{FABDEM_BASE}", "source_license": "CC BY-NC-SA 4.0",
                 "paper_ref": "Hawker et al. (2022), Environ. Res. Lett. 17, 024016",
                 "notes": f"virtual mosaic of {len(fabdem_tiles)} 1-degree tiles"},
    },
    "vito_lulc": {
        "path": _rel(lulc_tif), "data_type": "RasterDataset", "driver": "raster",
        "crs": 4326, "nodata": VITO_NODATA,
        "meta": {"category": "landuse", "unit": "-", "version": "v3.0.1 epoch 2019",
                 "source_url": VITO_URL, "source_license": "Copernicus, free and open",
                 "paper_ref": "Buchhorn et al. (2020), doi:10.5281/zenodo.3939050"},
    },
    "vito_reclass": {
        "path": _rel(lulc_csv), "data_type": "DataFrame", "driver": "csv",
        "driver_kwargs": {"index_col": "landuse"},
        "meta": {"category": "landuse",
                 "notes": "VITO class -> Manning's n (column N) and description",
                 "source_url": VITO_MAPPING_URL},
    },
    "hsg": {
        "path": _rel(soil_out["hsg_mdg.tif"]), "data_type": "RasterDataset", "driver": "raster",
        "crs": 4326, "nodata": 0,
        "meta": {"category": "soil", "unit": "1=A 2=B 3=C 4=D",
                 "source_url": "https://maps.isric.org (WCS 2.0.1)",
                 "source_license": "CC BY 4.0",
                 "paper_ref": "Poggio et al. (2021), SOIL 7, 217-240",
                 "notes": "derived from depth-weighted sand/clay via the USDA texture triangle"},
    },
}

for arc, path in gcn_clipped.items():
    entries[f"gcn250_{arc.lower()}"] = {
        "path": _rel(path), "data_type": "RasterDataset", "driver": "raster",
        "crs": 4326, "nodata": GCN250_NODATA,
        "meta": {"category": "soil", "unit": "-", "version": arc,
                 "source_url": f"https://doi.org/10.6084/m9.figshare.{GCN250_FIGSHARE_ID}",
                 "source_license": "CC BY 4.0",
                 "paper_ref": "Jaafar et al. (2019), Sci. Data 6, 145",
                 "notes": f"SCS curve number, antecedent runoff condition {arc[3:]}"},
    }

for path in event_files:
    entries[path.stem] = {
        "path": _rel(path), "data_type": "RasterDataset", "driver": "netcdf",
        "crs": 4326, "rename": {"Precipitation": "precip"},
        "meta": {"category": "meteo", "unit": "mm/hr",
                 "source_url": "GPM IMERG Final Run daily (GES DISC)",
                 "source_license": "NASA, free and open",
                 "notes": "rain-on-grid forcing, one historical tropical cyclone"},
    }

entries["design_depth_24h"] = {
    "path": _rel(idf_nc), "data_type": "RasterDataset", "driver": "netcdf",
    "crs": 4326,
    "meta": {"category": "meteo", "unit": "mm",
             "source_url": "GPM IMERG Final Run daily (GES DISC)",
             "notes": f"Gumbel 24 h design depth, return periods {list(RETURN_PERIODS)}; "
                      f"dimension 'return_period'"},
}

entries["pluvial_catchments_mdg"] = {
    "path": _rel(domain_gpkg), "data_type": "GeoDataFrame", "driver": "vector",
    "crs": 4326,
    "meta": {"category": "hydrography",
             "source_url": "https://www.hydrosheds.org/products/hydrobasins",
             "paper_ref": "Lehner & Grill (2013), Hydrol. Process. 27, 2171-2186",
             "notes": f"HydroBASINS v1c level 07, exploded, area >= "
                      f"{MIN_CATCHMENT_AREA_KM2:g} km2, {len(gdf_catch)} domains"},
}


def dump_yaml(obj, indent: int = 0) -> str:
    """Minimal YAML writer - avoids a PyYAML dependency for this one file."""
    pad = "  " * indent
    lines = []
    for key, val in obj.items():
        if isinstance(val, dict):
            lines.append(f"{pad}{key}:")
            lines.append(dump_yaml(val, indent + 1))
        elif isinstance(val, (list, tuple)):
            lines.append(f"{pad}{key}: [{', '.join(str(v) for v in val)}]")
        elif isinstance(val, str):
            safe = val.replace('"', "'")
            lines.append(f'{pad}{key}: "{safe}"')
        elif val is None:
            lines.append(f"{pad}{key}: null")
        else:
            lines.append(f"{pad}{key}: {val}")
    return "\n".join(lines)


header = (f"# HydroMT-SFINCS data catalogue - Madagascar national pluvial flood model\n"
          f"# generated by Scripts/{Path('pluvial_input_preprocessing.ipynb').name} on "
          f"{datetime.now(timezone.utc):%Y-%m-%d}\n"
          f"# paths are relative to this file\n"
          f"meta:\n  root: \".\"\n  version: \"1.0\"\n\n")
catalog_path.write_text(header + dump_yaml(entries) + "\n")
register_output(catalog_path, dataset="HydroMT-SFINCS data catalogue", source="derived",
                note=f"{len(entries)} entries")

print(f"wrote {catalog_path.relative_to(EXEC_DIR)} with {len(entries)} entries:")
for k, v in entries.items():
    print(f"  {k:<34} {v['data_type']:<14} {v['path']}")

## 5.2 Provenance manifest and output inventory

`provenance.json` records, for every file this notebook wrote: the dataset it belongs to, the source
URL or derivation, a note on the processing applied, the file size, a SHA-256 checksum and the
creation timestamp. That is what makes a result traceable back to an input months later — and what
lets a reviewer confirm they hold the same bytes.

Checksumming re-reads every output, so it is skipped for files above `CHECKSUM_LIMIT_MB` (the raw
FABDEM tiles and GCN250 globals) unless you ask for it.

In [ ]:
CHECKSUM_LIMIT_MB = 200

manifest_rows = []
for entry in OUTPUTS:
    path = EXEC_DIR / entry["path"]
    row = dict(entry)
    row["exists"] = path.exists()
    if path.exists() and path.is_file():
        row["size_mb"] = round(path.stat().st_size / 1e6, 2)
        row["sha256"] = (sha256(path) if row["size_mb"] <= CHECKSUM_LIMIT_MB
                         else "skipped (>%d MB)" % CHECKSUM_LIMIT_MB)
    manifest_rows.append(row)

provenance = {
    "notebook": "Scripts/pluvial_input_preprocessing.ipynb",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "exec_dir": str(EXEC_DIR),
    "python": sys.version.split()[0],
    "library_versions": dict({m.__name__: getattr(m, "__version__", "n/a")
                              for m in (np, pd, gpd, rasterio, xr)},
                             GDAL=rasterio.__gdal_version__),
    "configuration": {
        "aoi_bounds": list(AOI_BOUNDS),
        "work_crs": WORK_CRS,
        "min_catchment_area_km2": MIN_CATCHMENT_AREA_KM2,
        "n_catchments": int(len(gdf_catch)),
        "fabdem_tiles": len(fabdem_tiles),
        "gcn250_arcs": list(GCN250_ARCS),
        "soilgrids_depths": list(SOILGRIDS_DEPTHS),
        "tc_search_radius_km": TC_SEARCH_RADIUS_KM,
        "swath_radius_km": SWATH_RADIUS_KM,
        "exported_events": [r["event"] for r in event_rows],
        "return_periods": list(RETURN_PERIODS),
        "depth_duration_exponent": DD_EXPONENT,
    },
    "outputs": manifest_rows,
}

provenance_path = PROC_DIR / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2, default=str))

inventory = (pd.DataFrame(manifest_rows)[["dataset", "path", "size_mb", "exists"]]
             .sort_values(["dataset", "path"]).reset_index(drop=True))
print(f"{len(inventory)} outputs | {inventory['size_mb'].sum():,.0f} MB total")
print(f"processed/ : {sum(f.stat().st_size for f in PROC_DIR.rglob('*') if f.is_file()) / 1e6:,.0f} MB")
print(f"raw/       : {sum(f.stat().st_size for f in RAW_DIR.rglob('*') if f.is_file()) / 1e6:,.0f} MB")
print(f"manifest   : {provenance_path.relative_to(EXEC_DIR)}")
inventory

---

# 6. Next steps

The inputs are complete. What follows in the modelling chain:

1. **Per-catchment SFINCS builds** — loop over `catchment_id`, calling the HydroMT recipe in
   section 5.1 for each domain. Suggested grid: 50 m cells with `nr_subgrid_pixels=5`, so the
   subgrid tables see the full 30 m FABDEM detail.
2. **Roughness** — `setup_subgrid(datasets_rgh=[{"lulc": "vito_lulc", "reclass_table":
   "vito_reclass"}])` builds the Manning's *n* field from the two land-cover entries in the
   catalogue. Deliberately not pre-computed here (see *Deliberately not produced here* in the
   foreword).
3. **Observation points** — `setup_observation_points()` per domain, at the settlements, gauges and
   critical assets the results have to be reported on.
4. **Calibration** — run the exported historical events (section 4.5), compare modelled extents
   against reported flooding, and adjust roughness and the CN antecedent condition
   (`gcn250_arcii` → `gcn250_arciii`) rather than the topography.
5. **Hazard maps** — run the design hyetographs (section 4.6) per return period, after replacing the
   depth-duration exponent with a value calibrated on IMERG half-hourly data.

## Re-running this notebook

* **Everything cached:** run all cells — under three minutes, no downloads.
* **Refresh one dataset:** delete its outputs and re-run that section (`OVERWRITE` stays `False`).
* **Full rebuild:** set `OVERWRITE = True` in section 1.2. Budget ~1 hour and ~4 GB.
* **One environment does everything.** No CLIMADA and no `osgeo` bindings — see *Software
  environments* in the foreword. The GDAL command-line tools must be on `PATH`; section 1.1 checks
  and reports this.
* **Different AOI** (a province, a single catchment): change `MIN_CATCHMENT_AREA_KM2` or filter
  `gdf_catch` before section 2.1, and every static dataset follows the new `AOI_BOUNDS` automatically.